In [1]:
from config import init_env
from config import variables
import importlib
variables = importlib.reload(variables)
init_env.set_environment_variables()
import requests


In [2]:
import sys

import logging
# 1) 全局基础配置（只需执行一次）
logging.basicConfig(
    level=logging.INFO,  # 全局最低级别：DEBUG/INFO/WARNING/ERROR/CRITICAL
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    stream=sys.stdout,   # 输出到 Notebook 的输出区
    force=True           # 覆盖已有配置（Notebook重跑时很有用）
)

# 2) 获取你模块的 logger（与代码一致）
log = logging.getLogger(__name__)
log.setLevel(logging.INFO)  # 可按需调高/调低

# 3) 现在日志会显示在 Notebook 输出区
log.info("Logging is configured. You should see this message.")


from rich.console import Console
from rich.pretty import Pretty
console = Console(soft_wrap=True)  # 换行控制

2026-03-27 06:30:16,456 - __main__ - INFO - Logging is configured. You should see this message.


## Hana Database

### Create the connection to Hana database

In [2]:
from hdbcli import dbapi
# Initial the cursor
connection=dbapi.connect(
        address="e41c3eb7-55e9-47db-8915-e1ab64b9872a.hna0.prod-eu10.hanacloud.ondemand.com",
        port="443",
        user="USR_7K4HHI6O79L4LB691X7CN6MUQ",
        password="Jx8Q6g6l7NSZIMnooxdlCdkCLdEiR9--w4NSU4A0uZKfvLbrjKkgCJ9V2BXRHkQKP9ENitstBv4nv8.OraOa27HlVGwl50.D9BoaNPhoUDl-oGEbek1.n7QmYm-i3r.d",
        autocommit=True,
        sslValidateCertificate=False
    )

In [3]:
# (Optional)Check the connetion
connection.isconnected()

True

In [14]:
# Initial the cursor
cursor = connection.cursor() 

### HANA database table

#### Create a table

In [36]:
# Create a custom table with attribute

table_name = "LANGCHAIN_DEMO_SELF_QUERY"
try:
  cursor.execute(
      f'''
      CREATE TABLE "{table_name}" (
        "id"        INTEGER PRIMARY KEY,
        "name"      NVARCHAR(100),
        "is_active" BOOLEAN,
        "height"    DOUBLE,
        "VEC_TEXT"  NCLOB,
        "VEC_META"  NCLOB,
        "VEC_VECTOR" REAL_VECTOR(768)
      )
      '''
  )
  print(f'Table "{table_name}" created in the SAP HANA database.')

except Exception:
    print(f"Table  {table_name} already exsits.")
    pass

Table "LANGCHAIN_DEMO_SELF_QUERY" created in the SAP HANA database.


![](./images/TableCreation.png)

#### Delete a table

In [37]:
# Delete exsiting table if exists
try:
    cursor.execute(f"DROP TABLE {table_name}")
    print("Table dropped successfully.")
except Exception:
    print("No existing table.")
    pass

Table dropped successfully.


In [39]:
# Delete exsiting table if exists
try:
    cursor.execute(f"DROP TABLE {table_name}")
    print("Table dropped successfully.")
except Exception:
    print("No existing table.")
    pass

No existing table.


## Simple RAG Case: Local Document

### Prepare Document

In [8]:
# Step 1: Load documents

from langchain_community.document_loaders import PyPDFDirectoryLoader
DATA_PATH = r"datafiles"
loader = PyPDFDirectoryLoader(DATA_PATH)
documents = loader.load()
print(f"Loaded {len(documents)} documents.")

Loaded 6 documents.


In [9]:
# Step 2: Chunk documents

from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=150,
    length_function=len,
)
split_documents = text_splitter.split_documents(documents)
print(f"Split into {len(split_documents)} chunks.")

Split into 25 chunks.


### Embedding

In [13]:
# Step 3: Set embedding model  
 
from gen_ai_hub.proxy.langchain.openai import ChatOpenAI, OpenAIEmbeddings

embedding_model = OpenAIEmbeddings(
    deployment_id=variables.EMBEDDING_DEPLOYMENT_ID
    )  # Deployment ID of text-embedding-3-large

In [14]:
from langchain_hana import HanaDB

# Step 4: Define the embedding table 
table_name="TEST_EMBEDDING_TABLE"
db = HanaDB(
    embedding=embedding_model, 
    connection=connection, 
    table_name=table_name
)

# Step 5: Add embeded chunks into the table.  
db.add_documents(split_documents)
print(f"Table {db.table_name} created in the SAP HANA database.")

2026-03-10 05:11:33,293 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/d12a770eac7c91c9/embeddings?api-version=2025-03-01-preview "HTTP/1.1 200 OK"
2026-03-10 05:11:33,709 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/d12a770eac7c91c9/embeddings?api-version=2025-03-01-preview "HTTP/1.1 200 OK"
Table TEST_EMBEDDING_TABLE created in the SAP HANA database.


<i><b>db.add</b></i> will create the table using the given table name if it does not exsit.
The table is defaulted with 3 fields:
* A column VEC_TEXT, which contains the text of the Document.
* A column VEC_META, which contains the metadata of the Document.
* A column VEC_VECTOR, which contains the embeddings-vector of the Document’s text.

![](./images/db_add.png)</p>
If the table already exsits, <i><b>db.add</b></i> will add new entries into the table.
![](./images/DuplicateEntries.png)</p>

In [ ]:
# (Optional) delete this table for repeating test
try:
    cursor.execute(f'DROP TABLE "{table_name}"')
    print(f'Table "{table_name}" dropped successfully.')
except Exception:
    print(f'Table "{table_name}" does not exist.')


Table "TEST_EMBEDDING_TABLE" dropped successfully.


### Check the embeddings in SAP HANA Cloud Vector Engine 

In [ ]:
from IPython.display import Markdown
 
# Use `db.table_name` instead of `variables.EMBEDDING_TABLE` because HANA driver sanitizes a table name by removing unaccepted characters
is_ok = cursor.execute(
    f'''
    SELECT "VEC_TEXT", "VEC_META", TO_NVARCHAR("VEC_VECTOR") FROM "{table_name}"
    ''')

record_columns=cursor.fetchone()

if record_columns:
    display({"VEC_TEXT" : record_columns[0], "VEC_META" : eval(record_columns[1]), "VEC_VECTOR" : record_columns[2]})


{'VEC_TEXT': 'Introduction \nWe SAP are excited to announce that we have started working on a VS Code extension for \nABAP . We understand that the community has high expectations, and we want to \ncommunicate transparently about what you can expect from ABAP Development Tools for \nVS Code. In this article, we will share details about the scope of the ﬁrst release and what’s \nplanned for the future. \nSee related article: Behind the Design: How We Transformed the ABAP Development Tools',
 'VEC_META': {'producer': 'Microsoft: Print To PDF',
  'creator': 'PyPDF',
  'creationdate': '2025-11-18T10:40:21+08:00',
  'author': 'SUN Yufeng (BD/PTD-SPR1)',
  'moddate': '2025-11-18T10:40:21+08:00',
  'title': 'Microsoft Word - ABAP Development Tools for VS Code Everything You Need to Know .docx',
  'source': 'datafiles/ABAP Development Tools for VS Code Everything You Need to Know.pdf',
  'total_pages': 2,
  'page': 0,
  'page_label': '1'},
 'VEC_VECTOR': '[0.011229125,-0.02020525,-0.009772568,

### Run RAG

In [68]:
from gen_ai_hub.proxy.langchain.init_models import init_llm
model = init_llm(
    'gpt-4o', 
    temperature=0.1,
    max_tokens=8000
)


In [ ]:
# Alternatively
model = ChatOpenAI(
    deployment_id=variables.LLM_DEPLOYMENT_ID
)  # LLM deployment ID. Here gpt-4o has been maintained

In [69]:
# Create a retriever instance of the vector store
retriever = db.as_retriever(
    search_kwargs={"k": 1}
)

In [ ]:

from langchain.chains import RetrievalQA
# Create the QA instance to query llm based on custom documents
qa = RetrievalQA.from_llm(
    llm=model, 
    retriever=retriever, 
    return_source_documents=True)

# Send query
query = "Why is ABAP in VS Code is so appealing?"

answer = qa.invoke(query)
display(answer["result"])


'ABAP in Visual Studio Code is appealing because it provides more flexibility in choosing a development environment, addressing the demand from users for support beyond SAP GUI and Eclipse. Visual Studio Code is a popular IDE known for its versatility, ease of use, and extensive range of extensions, which can enhance the development experience compared to existing tools. Additionally, it aligns with the preferences of a significant number of users based on survey results from 2023 and 2025.'

In [ ]:

# Check addition information from retrievr 
for document in answer['source_documents']:
    display(document.metadata)   
    print(document.page_content)



{'producer': 'Microsoft: Print To PDF',
 'creator': 'PyPDF',
 'creationdate': '2025-11-18T10:42:07+08:00',
 'author': 'SUN Yufeng (BD/PTD-SPR1)',
 'moddate': '2025-11-18T10:42:07+08:00',
 'title': 'Microsoft Word - ABAP Development Tools for VS Code Everything You Need to Know .docx',
 'source': 'datafiles/Behind the Design How We Transformed the ABAP Development Tools Architecture to Support More IDEs.pdf',
 'total_pages': 4,
 'page': 0,
 'page_label': '1'}

Introduction 
Currently, oƯicial ABAP tool support exists for SAP GUI and Eclipse. For years, users 
have asked us to bring this support to additional IDEs. Based on the user survey 
results from 2023 and 2025, the most requested development environment is Visual 
Studio Code (VS Code). However, many users have also expressed interest in other 
environments such as JetBrains IDEs, Neovim, or even Zed. In short, our user base 
wants more ﬂexibility when choosing their development environment.


## RAG Case: Online Document

In [110]:
TABLE_NAME = "GIT_DOCS"
LLM_MODEL_NAME = 'gpt-4o'
EMBEDDINGS_MODEL_NAME ='text-embedding-3-large'
GIT_URL="https://github.com/SAP/terraform-provider-btp"

### Prepare Document

#### Load document from git repository

In [111]:
from langchain_community.document_loaders import GitLoader

# Define a function to fetch file from a git respository
def fetch_gitrepository_docs(gitrepository_url):
    try:
        log.info("Getting the documents from the GitHub repository: %s", gitrepository_url)
        loader = GitLoader(
            clone_url=gitrepository_url,
            repo_path="./gen/docs/",
            file_filter=lambda file_path: file_path.startswith("./gen/docs/docs")
            and file_path.endswith(".md"),
            branch="main",
        )
        documents = loader.load()
        log.info("Documents loaded successfully. count=%d", len(documents) if documents else 0)
        return documents
    except Exception as e:
        log.error(f"Error occurred while loading documents: {str(e)}")


In [112]:
# Test the function
fetch_gitrepository_docs(gitrepository_url=GIT_URL)

2026-03-11 06:56:30,214 - __main__ - INFO - Getting the documents from the GitHub repository: https://github.com/SAP/terraform-provider-btp
2026-03-11 06:56:32,924 - __main__ - INFO - Documents loaded successfully. count=128


[Document(metadata={'source': 'docs/index.md', 'file_path': 'docs/index.md', 'file_name': 'index.md', 'file_type': '.md'}, page_content='---\npage_title: "SAP BTP Provider"\nsubcategory: ""\ndescription: |-\n  The Terraform provider for SAP BTP enables you to automate the provisioning, management, and configuration of resources on SAP Business Technology Platform https://account.hana.ondemand.com/. By leveraging this provider, you can simplify and streamline the deployment and maintenance of BTP services and applications.\n---\n# Terraform Provider for SAP BTP\n\nThe Terraform provider for SAP BTP enables you to automate the provisioning, management, and configuration of resources on [SAP Business Technology Platform](https://account.hana.ondemand.com/). By leveraging this provider, you can simplify and streamline the deployment and maintenance of BTP services and applications.\n\n## Example Usage\n\n```terraform\nterraform {\n  required_providers {\n    btp = {\n      source  = "SAP/b

#### Chunk the document 

In [113]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document

def split_docs_into_chunks(
    documents: list[Document], chunk_size: int = 1000, chunk_overlap: int = 100
):
    """
    Splits a list of documents into chunks of specified size with overlap.

    Args:
        documents (list[Document]): The list of documents to be split into chunks.
        chunk_size (int, optional): The size of each chunk. Defaults to 1000.
        chunk_overlap (int, optional): The overlap between consecutive chunks. Defaults to 100.

    Returns:
        list[list[Document]]: A list of chunks, where each chunk is a list of documents.

    """
    try:
        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            length_function=len,
            add_start_index=True,
        )
        chunks = text_splitter.split_documents(documents)
        log.info(f"Split {len(documents)} documents into {len(chunks)} chunks.")

        return chunks
    except Exception as e:
        log.error(f"An error occurred while splitting documents into chunks: {str(e)}")
        raise

In [114]:
# Test the function
git_docs=fetch_gitrepository_docs(gitrepository_url=GIT_URL)
chunks=split_docs_into_chunks(git_docs)

2026-03-11 06:56:40,500 - __main__ - INFO - Getting the documents from the GitHub repository: https://github.com/SAP/terraform-provider-btp
2026-03-11 06:56:42,950 - __main__ - INFO - Documents loaded successfully. count=128
2026-03-11 06:56:42,960 - __main__ - INFO - Split 128 documents into 516 chunks.


### Prepare language model

In [6]:
from gen_ai_hub.proxy.langchain.openai import ChatOpenAI
from gen_ai_hub.proxy.langchain.openai import OpenAIEmbeddings
from gen_ai_hub.proxy.core.proxy_clients import get_proxy_client
from langchain_core.rate_limiters import InMemoryRateLimiter

# Define a function to define the  chat llm and embedding model to be used
def create_llm_and_embeddings(llm_model,embedding_model):
    
    # Get the proxy client for the AI Core service
    proxy_client = get_proxy_client("gen-ai-hub")

    rate_limiter = InMemoryRateLimiter(
        requests_per_second=0.5,  # We can only make a request once every 5 seconds
        check_every_n_seconds=0.1,  # Wake up every 100 ms to check whether allowed to make a request,
        max_bucket_size=10,  # Controls the maximum burst size.
    )

    llm = ChatOpenAI(
        proxy_model_name=llm_model,
        proxy_client=proxy_client,
        temperature=0.1,
        rate_limiter=rate_limiter,
    )

    embeddings = OpenAIEmbeddings(
        proxy_model_name=embedding_model,
        proxy_client=proxy_client,
        show_progress_bar=True,
    )
    return llm, embeddings

2026-03-11 01:48:31,044 - numexpr.utils - INFO - NumExpr defaulting to 8 threads.


In [51]:
# Test the function
llm,embeddings=create_llm_and_embeddings(
    llm_model=LLM_MODEL_NAME,
    embedding_model=EMBEDDINGS_MODEL_NAME,
)

print(llm)
print(embeddings)

rate_limiter=<langchain_core.rate_limiters.InMemoryRateLimiter object at 0x7fd9d9e4add0> client=<gen_ai_hub.proxy.native.openai.clients.ChatCompletions object at 0x7fd9cb6948d0> async_client=<gen_ai_hub.proxy.native.openai.clients.AsyncChatCompletions object at 0x7fd9cb695ed0> root_client=<gen_ai_hub.proxy.native.openai.clients.OpenAI object at 0x7fd9cb657ed0> root_async_client=<gen_ai_hub.proxy.native.openai.clients.AsyncOpenAI object at 0x7fd9d9eb87d0> model_name='gpt-4o' temperature=0.1 model_kwargs={} openai_api_key=SecretStr('**********') n=1 proxy_client=GenAIHubProxyClient(base_url=None, auth_url=None, client_id=None, client_secret=None, resource_group=None, ai_core_client=<ai_core_sdk.ai_core_v2_client.AICoreV2Client object at 0x7fd9db1ab0e0>) deployment_id='da74d08d2f277477' config_name='islm.dpl.cfg.ZTEST_GENAI_OPENAI.7AB55BB2EC021FD186C6F4447E709D45.Deployed based on model ZGPTOPENAI4 training 1' config_id='7eb56362-ff64-421e-9f91-cb765773ddf9' proxy_model_name='gpt-4o'
clie

### Embedding

#### Delete existing table content

In [123]:
# Define a function to check whether a table exists or not
def check_if_exists(table_name, schema_name="USR_7K4HHI6O79L4LB691X7CN6MUQ"):
    connection_to_hana = init_env.get_connection_to_hana_db()
    cursor = connection_to_hana.cursor()

    # Check if the table exists
    check_table_query ="""
    SELECT COUNT(*)
    FROM TABLES
    WHERE SCHEMA_NAME = ? AND TABLE_NAME = ?
    """

    cursor.execute(check_table_query, (schema_name, table_name))
    return cursor.fetchone()[0] > 0


In [98]:
# Test the function
check_if_exists(table_name=TABLE_NAME)

True

In [124]:
# Define a function to remove existing table
def teardown_hana_table(table_name):
    
    exists = check_if_exists(table_name=table_name)
    if exists is False:
        log.info(f"Table {table_name} does not exsit. Nothing to clean up.")
        return
    
    try:
        connection_to_hana = init_env.get_connection_to_hana_db()
        cursor = connection_to_hana.cursor()
        log.info(f"Dropping table {table_name}")
        cursor.execute(f"DROP TABLE {table_name}")
        cursor.close()
        log.info(f"Table {table_name} dropped successfully.")
    except Exception as e:
        log.error(type(e))
        log.error(f"Error dropping table: {str(e)}")


In [95]:
# Test the function
teardown_hana_table(table_name=TABLE_NAME)

2026-03-10 06:05:06,875 - __main__ - INFO - Table GIT_DOCS does not exsit. Nothing to clean up


#### Embedding and save to HANA table

In [125]:
from langchain_hana import HanaDB

# Define a function to do the embedding and then save the data into HANA table
def ingest(documents):
    try:
        teardown_hana_table(table_name=TABLE_NAME)
        log.info(f"Start ingesting data in {TABLE_NAME}")
        
        connection_to_hana = init_env.get_connection_to_hana_db()
        cursor = connection_to_hana.cursor()
        
        chunks = split_docs_into_chunks(documents=documents)

        _, embeddings = create_llm_and_embeddings(
            llm_model=LLM_MODEL_NAME,
            embedding_model=EMBEDDINGS_MODEL_NAME,
        )


        db = HanaDB(
            embedding=embeddings, 
            connection=connection_to_hana, 
            table_name=TABLE_NAME
        )
       
        log.info("Adding documents chunks to the HANA DB")
        db.add_documents(chunks)
        log.info("Documents added successfully.")
        log.info("Ingestion completed successfully.")

        cursor.close()
    except Exception as e:
        log.error(f"Error occurred during ingestion: {str(e)}")

In [126]:
# Run the ingest
git_docs=fetch_gitrepository_docs(gitrepository_url=GIT_URL)
ingest(documents=git_docs)


2026-03-10 07:04:36,259 - __main__ - INFO - Getting the documents from the GitHub repository: https://github.com/SAP/terraform-provider-btp
2026-03-10 07:04:39,326 - __main__ - INFO - Documents loaded successfully. count=128
2026-03-10 07:04:39,748 - __main__ - INFO - Dropping table GIT_DOCS
2026-03-10 07:04:39,763 - __main__ - INFO - Table GIT_DOCS dropped successfully.
2026-03-10 07:04:39,768 - __main__ - INFO - Start ingesting data in GIT_DOCS
2026-03-10 07:04:39,964 - __main__ - INFO - Split 128 documents into 516 chunks.
2026-03-10 07:04:40,313 - __main__ - INFO - Adding documents chunks to the HANA DB


  0%|          | 0/33 [00:00<?, ?it/s]

2026-03-10 07:04:40,870 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/d12a770eac7c91c9/embeddings?api-version=2025-03-01-preview "HTTP/1.1 200 OK"


  3%|▎         | 1/33 [00:00<00:16,  1.93it/s]

2026-03-10 07:04:41,501 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/d12a770eac7c91c9/embeddings?api-version=2025-03-01-preview "HTTP/1.1 200 OK"


...

2026-03-10 07:04:57,915 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/d12a770eac7c91c9/embeddings?api-version=2025-03-01-preview "HTTP/1.1 200 OK"


 97%|█████████▋| 32/33 [00:17<00:00,  1.87it/s]

2026-03-10 07:04:58,257 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/d12a770eac7c91c9/embeddings?api-version=2025-03-01-preview "HTTP/1.1 200 OK"


100%|██████████| 33/33 [00:17<00:00,  1.84it/s]


2026-03-10 07:04:58,483 - __main__ - INFO - Documents added successfully.
2026-03-10 07:04:58,484 - __main__ - INFO - Ingestion completed successfully.


### Run RAG

In [ ]:
from langchain_hana import HanaDB

# Create a retriever

llm, embeddings = create_llm_and_embeddings(
    llm_model=LLM_MODEL_NAME,
    embedding_model=EMBEDDINGS_MODEL_NAME,
)

connection_to_hana = init_env.get_connection_to_hana_db()

db = HanaDB(
    embedding=embeddings, 
    connection=connection_to_hana, 
    table_name=TABLE_NAME
)
retriever = db.as_retriever(search_kwargs={"k":8 })

from langchain.chains import RetrievalQA

# Create the QA instance to query llm based on custom documents
qa = RetrievalQA.from_llm(
    llm=llm, 
    retriever=retriever, 
    return_source_documents=True)



Test RAG. The source of the answer origianlly comes from:</br>
https://github.com/SAP/terraform-provider-btp/blob/main/docs/data-sources/subaccount_destination_trust.md

In [21]:
# Write a question
query = "三国演义中的刘备是什么人?"
#query = "What is btp_subaccount_destination_trust and how to use it?"
 
# Answer without RAG
answer = llm.invoke(query)
print(answer.content)

2026-03-11 02:32:02,953 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/da74d08d2f277477/chat/completions?api-version=2025-03-01-preview "HTTP/1.1 200 OK"
刘备是中国古典小说《三国演义》中的重要人物之一。他是蜀汉的建立者和第一任皇帝，字玄德，涿郡涿县（今河北省涿州市）人。刘备以仁德著称，常被描绘为一位有理想、有抱负的君主，尽管出身寒微，但他以仁义和宽厚待人，赢得了许多贤才的支持。

在《三国演义》中，刘备与关羽、张飞结为兄弟，三人桃园结义的故事广为流传。刘备一生致力于匡扶汉室，恢复汉朝的统治。他在诸葛亮等人的辅佐下，逐步建立了蜀汉政权，与曹操的魏国和孙权的吴国形成三足鼎立的局面。

尽管刘备在历史上和小说中都被视为一位仁君，但他也有优柔寡断的一面，尤其是在用人和决策方面。《三国演义》通过刘备的故事，展现了忠义、仁德以及乱世中的政治斗争。


In [20]:
# Answer with RAG 
answer = qa.invoke(query)
print(answer["result"])

  0%|          | 0/1 [00:00<?, ?it/s]

2026-03-11 02:31:53,162 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/d12a770eac7c91c9/embeddings?api-version=2025-03-01-preview "HTTP/1.1 200 OK"


100%|██████████| 1/1 [00:00<00:00,  4.55it/s]


2026-03-11 02:31:54,923 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/da74d08d2f277477/chat/completions?api-version=2025-03-01-preview "HTTP/1.1 200 OK"
刘备是中国古典小说《三国演义》中的重要角色之一。他是蜀汉的开国皇帝，字玄德，汉景帝之后裔。刘备以仁德著称，重视人才，广纳贤士。他与关羽、张飞结为兄弟，共同建立了蜀汉政权。在《三国演义》中，刘备被描绘为一位有理想、有抱负的君主，尽管在政治和军事上面临诸多挑战，但始终坚持自己的信念。


## Advanced RAG: Self Query

In [108]:
from gen_ai_hub.proxy.langchain.openai import ChatOpenAI
from gen_ai_hub.proxy.langchain.openai import OpenAIEmbeddings
from gen_ai_hub.proxy.core.proxy_clients import get_proxy_client
from langchain_core.rate_limiters import InMemoryRateLimiter

# Define a function to define the  chat llm and embedding model to be used
def create_llm_and_embeddings(llm_model,embedding_model):
    
    # Get the proxy client for the AI Core service
    proxy_client = get_proxy_client("gen-ai-hub")
     
    rate_limiter = InMemoryRateLimiter(
        requests_per_second=0.5,  # We can only make a request once every 5 seconds
        check_every_n_seconds=0.1,  # Wake up every 100 ms to check whether allowed to make a request,
        max_bucket_size=10,  # Controls the maximum burst size.
    )

    llm = ChatOpenAI(
        proxy_model_name=llm_model,
        proxy_client=proxy_client,
        temperature=0.1,
        rate_limiter=rate_limiter,
    )

    embeddings = OpenAIEmbeddings(
        proxy_model_name=embedding_model,
        proxy_client=proxy_client,
        show_progress_bar=True,
    )
    return llm, embeddings

# Initial the model
LLM_MODEL_NAME = 'gpt-4o'
EMBEDDINGS_MODEL_NAME ='text-embedding-3-large'
llm, embeddings = create_llm_and_embeddings(
    llm_model=LLM_MODEL_NAME,
    embedding_model=EMBEDDINGS_MODEL_NAME,
)


# Define a function to check whether a table exists or not
def check_if_exists(table_name, schema_name="USR_7K4HHI6O79L4LB691X7CN6MUQ"):
    connection_to_hana = init_env.get_connection_to_hana_db()
    cursor = connection_to_hana.cursor()

    # Check if the table exists
    check_table_query ="""
    SELECT COUNT(*)
    FROM TABLES
    WHERE SCHEMA_NAME = ? AND TABLE_NAME = ?
    """

    cursor.execute(check_table_query, (schema_name, table_name))
    return cursor.fetchone()[0] > 0

# Define a function to remove existing table
def teardown_hana_table(table_name):
    
    exists = check_if_exists(table_name=table_name)
    if exists is False:
        log.info(f"Table {table_name} does not exsit. Nothing to clean up.")
        return
    
    try:
        connection_to_hana = init_env.get_connection_to_hana_db()
        cursor = connection_to_hana.cursor()
        log.info(f"Dropping table {table_name}")
        cursor.execute(f"DROP TABLE {table_name}")
        cursor.close()
        log.info(f"Table {table_name} dropped successfully.")
    except Exception as e:
        log.error(type(e))
        log.error(f"Error dropping table: {str(e)}")

### Prepare Document

#### Document from online PDF

In [15]:
# PDF data
episodes = [
    {
        "url": "https://sap-podcast-bucket.s3.amazonaws.com/the-future-of-supply-chain/The_Future_of_Supply_Chain_Episode_64_transcript.pdf",
        "title": "Future of Supply Chain: Episode 64: Proactively Planning for Risk in Your Supply Chain with Everstream's Koray Kose and SAP's Volker Wilhelm",
        "episode": 64,
    },
    {
        "url": "https://sap-podcast-bucket.s3.amazonaws.com/the-future-of-supply-chain/The_Future_of_Supply_Chain_Episode_65_transcript.pdf",
        "title": "The Future of Supply Chain: Episode 65: Grounding Your Supply Chain in Data with Google Cloud’s Paula Natoli",
        "episode": 65,
    },
]

web_pdf1 = {
    "url": "https://sap-podcast-bucket.s3.amazonaws.com/the-future-of-supply-chain/The_Future_of_Supply_Chain_Episode_64_transcript.pdf",
    "title": "Future of Supply Chain: Episode 64: Proactively Planning for Risk in Your Supply Chain with Everstream's Koray Kose and SAP's Volker Wilhelm",
    "episode": 64,
} 

# Initiate the HanaDB using the table
PDF_TABLE_NAME="PDF_DOCS"
from langchain_hana import HanaDB
db = HanaDB(
        embedding=embeddings, 
        connection= init_env.get_connection_to_hana_db(), 
        table_name=PDF_TABLE_NAME
)

##### Load the document 

In [175]:
# Create a function to load one online PDF
from langchain_community.document_loaders import PyMuPDFLoader

def fetch_web_pdf(web_pdf):
    
    try:
        loader = PyMuPDFLoader(web_pdf["url"])
        documents = loader.load()
        #Add 2 new metadata
        for doc in documents:
            doc.metadata["podcast_title"] = web_pdf["title"]
            doc.metadata["episode"] =  web_pdf["episode"]
    
        log.info("PDF <%s> with %d pages loaded successfully.", web_pdf["title"],len(documents))
                
        return documents

    except Exception as e:
        log.error(f"Error during fetching PDF: {str(e)}")

In [ ]:
#Test 
documents=fetch_web_pdf(web_pdf=web_pdf1)

2026-03-11 08:32:29,982 - __main__ - INFO - PDF <Future of Supply Chain: Episode 64: Proactively Planning for Risk in Your Supply Chain with Everstream's Koray Kose and SAP's Volker Wilhelm> with 9 pages loaded successfully.


##### Chunk the document 

PyMuPDFLoader by default load PDF page by page. Try to skip this step.


Optional:<br>
<i>from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

split_docs = splitter.split_documents(documents)


##### Embedding the document 

In [ ]:
# Can be simply done as below
db.add_documents(documents)

  0%|          | 0/1 [00:00<?, ?it/s]

2026-03-11 08:14:49,316 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/d12a770eac7c91c9/embeddings?api-version=2025-03-01-preview "HTTP/1.1 200 OK"


100%|██████████| 1/1 [00:00<00:00,  1.13it/s]


[]

In [ ]:
# Define a function to do the embedding and then save the data into HANA table
def ingest_pdf(pdf_list):
    try:
        teardown_hana_table(table_name=PDF_TABLE_NAME)
        log.info(f"Start ingesting data in {PDF_TABLE_NAME}")
        
        db = HanaDB(
            embedding=embeddings, 
            connection= init_env.get_connection_to_hana_db(), 
            table_name=PDF_TABLE_NAME
        )

        for ep in pdf_list:
            documents=fetch_web_pdf(web_pdf=ep)
            db.add_documents(documents)
            log.info("Documents added successfully.")
       
        log.info("Ingestion completed successfully.")
 
    except Exception as e:
        log.error(f"Error occurred during ingestion: {str(e)}")

In [ ]:
# Run the ingest
ingest_pdf(pdf_list=episodes)

2026-03-11 09:04:30,811 - __main__ - INFO - Table PDF_DOCS does not exsit. Nothing to clean up.
2026-03-11 09:04:30,813 - __main__ - INFO - Start ingesting data in PDF_DOCS


2026-03-11 09:04:31,251 - __main__ - INFO - PDF <Future of Supply Chain: Episode 64: Proactively Planning for Risk in Your Supply Chain with Everstream's Koray Kose and SAP's Volker Wilhelm> with 9 pages loaded successfully.


  0%|          | 0/1 [00:00<?, ?it/s]

2026-03-11 09:04:32,125 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/d12a770eac7c91c9/embeddings?api-version=2025-03-01-preview "HTTP/1.1 200 OK"


100%|██████████| 1/1 [00:00<00:00,  1.14it/s]

...

2026-03-11 09:04:32,344 - __main__ - INFO - PDF <The Future of Supply Chain: Episode 65: Grounding Your Supply Chain in Data with Google Cloud’s Paula Natoli> with 8 pages loaded successfully.


  0%|          | 0/1 [00:00<?, ?it/s]

2026-03-11 09:04:33,100 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/d12a770eac7c91c9/embeddings?api-version=2025-03-01-preview "HTTP/1.1 200 OK"


100%|██████████| 1/1 [00:00<00:00,  1.32it/s]

2026-03-11 09:04:33,124 - __main__ - INFO - Documents added successfully.
2026-03-11 09:04:33,124 - __main__ - INFO - Ingestion completed successfully.


### Self Query

In [4]:
from typing import Optional
from pydantic import BaseModel, Field
from langchain_core.prompts import PromptTemplate

# Define a function to generate a metadata filter from the origianl question
def extract_podcast_title_from_question(llm, question):
    class Podcast(BaseModel):
        title: Optional[str] = Field(
            default=None, description="The title or the episode number of the podcast"
        )

    # Define the prompt to extract data from user query
    system_template = """
    You are an expert extraction algorithm.
    Only extract relevant information from the question below.
    If you do not know the value of an attribute asked to extract,
    return null for the attribute's value.

    Text: {question}
    """

    prompt = PromptTemplate(template=system_template, input_variables=["question"])

    runnable = prompt | llm.with_structured_output(schema=Podcast)
    podcast = runnable.invoke({"question": question})
    print("Podcast Title:", podcast.title)
    return podcast.title

In [ ]:
#Test
question = "What is the summary of the episode 65?"
podcast = extract_podcast_title_from_question(llm, question)

# Show the filter
advanced_db_filter = {"title": {"$like": f"%{podcast.title()}%"}}
print(advanced_db_filter)

2026-03-16 07:53:47,400 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/da74d08d2f277477/chat/completions?api-version=2025-03-01-preview "HTTP/1.1 200 OK"
Podcast Title: Episode 65
Episode 65
{'title': {'$like': '%Episode 65%'}}


In [ ]:
#Create a funtion to do query using filter 
from langchain.chains import RetrievalQA

# Generate the answer to the question using retrieved documents and applied filters
def qa_documents_with_filters(db, llm, question, advanced_db_filter=None):
    qa_prompt_template = """
    You are an expert in SAP podcasts topics. You are provided multiple context items that are related to the prompt you have to answer.
    Use the following pieces of context to answer the question at the end.

    '''
    {context}
    '''

    Question: {question}
    """

    prompt = PromptTemplate(
        template=qa_prompt_template, input_variables=["context", "question"]
    )

    qa_chain = RetrievalQA.from_chain_type(
        llm,
        chain_type="stuff",
        retriever=db.as_retriever(search_kwargs={"k": 5, "filter": advanced_db_filter}),
        return_source_documents=True,
        verbose=True,
        chain_type_kwargs={"prompt": prompt},
    )

    result = qa_chain.invoke({"query": question})

    print("Source Documents:")
    for doc in result["source_documents"]:
        print("Title:", doc.metadata["title"], " Page Number:", doc.metadata["page"])

    print("Result:", result["result"])

In [17]:
#Test
question = "What is the summary of the episode 65?"
qa_documents_with_filters(db, llm, question)



> Entering new RetrievalQA chain...


  0%|          | 0/1 [00:00<?, ?it/s]

2026-03-16 07:59:04,517 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/d12a770eac7c91c9/embeddings?api-version=2025-03-01-preview "HTTP/1.1 200 OK"


100%|██████████| 1/1 [00:00<00:00,  3.73it/s]


2026-03-16 07:59:06,810 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/da74d08d2f277477/chat/completions?api-version=2025-03-01-preview "HTTP/1.1 200 OK"

> Finished chain.
Source Documents:
Title: Microsoft Word - Word - Transcript - Episode 64.docx  Page Number: 8
Title: Microsoft Word - Word - Transcript - Episode 64.docx  Page Number: 0
Title: Microsoft Word - Word - Transcript - Episode 65.docx  Page Number: 7
Title: Microsoft Word - Word - Transcript - Episode 65.docx  Page Number: 0
Title: Microsoft Word - Word - Transcript - Episode 65.docx  Page Number: 3
Result: Episode 65 of the "Future of Supply Chain" podcast features a discussion with Google Cloud's Paula Natoli, focusing on the importance of grounding supply chains in data rather than intuition. Paula emphasizes the need for supply chains to be powered by information and technology, highlighting the role of data in improving visibility and decision-m

In [ ]:
#Create a funtion to query the same question w/o filtering
def self_query(db, llm, question):
    print("Question: ", question)
    print("Without database filtering based on user query")
    qa_documents_with_filters(db, llm, question)

    print("With database filtering based on user query")
    podcast = extract_podcast_title_from_question(llm, question)

    advanced_db_filter = {"title": {"$like": f"%{podcast.title()}%"}}
    qa_documents_with_filters(db, llm, question, advanced_db_filter)

In [ ]:
#Test
question = "What is the summary of the episode 65?"

self_query(db, llm, question)

Question:  What is the summary of the episode 65?
Without database filtering based on user query


> Entering new RetrievalQA chain...


  0%|          | 0/1 [00:00<?, ?it/s]

2026-03-13 08:31:37,387 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/d12a770eac7c91c9/embeddings?api-version=2025-03-01-preview "HTTP/1.1 200 OK"


100%|██████████| 1/1 [00:00<00:00,  3.47it/s]


2026-03-13 08:31:39,728 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/da74d08d2f277477/chat/completions?api-version=2025-03-01-preview "HTTP/1.1 200 OK"

> Finished chain.
Source Documents:
Title: Microsoft Word - Word - Transcript - Episode 64.docx  Page Number: 8
Title: Microsoft Word - Word - Transcript - Episode 64.docx  Page Number: 0
Title: Microsoft Word - Word - Transcript - Episode 65.docx  Page Number: 7
Title: Microsoft Word - Word - Transcript - Episode 65.docx  Page Number: 0
Title: Microsoft Word - Word - Transcript - Episode 65.docx  Page Number: 3
Result: Episode 65 of the "Future of Supply Chain" podcast features a discussion with Google Cloud's Paula Natoli, focusing on the importance of grounding supply chains in data. Paula emphasizes that supply chains should be driven by information rather than intuition, highlighting the role of technology in powering supply chains. The conversation covers t

  0%|          | 0/1 [00:00<?, ?it/s]

2026-03-13 08:31:41,712 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/d12a770eac7c91c9/embeddings?api-version=2025-03-01-preview "HTTP/1.1 200 OK"


100%|██████████| 1/1 [00:00<00:00,  4.01it/s]

2026-03-13 08:31:41,717 - langchain_hana.vectorstores.create_where_clause - WARNING - Plain SQL Placeholder '?' for value='%Episode 65%'


2026-03-13 08:31:46,943 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/da74d08d2f277477/chat/completions?api-version=2025-03-01-preview "HTTP/1.1 200 OK"

> Finished chain.
Source Documents:
Title: Microsoft Word - Word - Transcript - Episode 65.docx  Page Number: 7
Title: Microsoft Word - Word - Transcript - Episode 65.docx  Page Number: 0
Title: Microsoft Word - Word - Transcript - Episode 65.docx  Page Number: 3
Title: Microsoft Word - Word - Transcript - Episode 65.docx  Page Number: 6
Title: Microsoft Word - Word - Transcript - Episode 65.docx  Page Number: 1
Result: Episode 65 of "The Future of Supply Chain" podcast, titled "Grounding Your Supply Chain in Data with Google Cloud’s Paula Natoli," focuses on the importance of data-driven decision-making in supply chains. Paula Natoli, from Google Cloud's Global Strategic Industries organization, discusses how supply chains are increasingly complex and cannot rel

### HANA Self Query

In [6]:
from langchain.chains import RetrievalQA
from langchain.chains.query_constructor.base import AttributeInfo
from langchain.prompts.chat import (
    ChatPromptTemplate,
    HumanMessagePromptTemplate,
    SystemMessagePromptTemplate,
)

from langchain.retrievers.self_query.base import SelfQueryRetriever
from langchain_hana import HanaTranslator
from langchain_hana import HanaDB

PDF_TABLE_NAME="PDF_DOCS"
db = HanaDB(
        embedding=embeddings, 
        connection= init_env.get_connection_to_hana_db(), 
        table_name=PDF_TABLE_NAME
)


In [7]:
def qa_documents_with_filters(db, llm, question, advanced_db_filter=None):
    """
    Generate the answer to the question using retrieved documents and applied filters.

    Args:
        db: The database object.
        llm: The language model object.
        question: The question string.
        advanced_db_filter: Optional filter to apply to the database query.
    """
    try:
        system_template = """
        You are an expert in SAP podcasts topics. You are provided multiple context items that are related to the 
        prompt you have to answer.
        Use the following pieces of context to answer the question at the end.
        '''
        {context}
        '''
        """

        human_template = "{question}"

        messages = [
            SystemMessagePromptTemplate.from_template(system_template),
            HumanMessagePromptTemplate.from_template(human_template),
        ]

        prompt = ChatPromptTemplate.from_messages(messages)

        qa_chain = RetrievalQA.from_chain_type(
            llm,
            chain_type="stuff",
            retriever=db.as_retriever(
                search_kwargs={"k": 5, "filter": advanced_db_filter}
            ),
            return_source_documents=True,
            verbose=True,
            chain_type_kwargs={"prompt": prompt},
        )

        result = qa_chain.invoke({"query": question})

        log.info("Source Documents:")
        for doc in result["source_documents"]:
            log.info(
                f"Title: {doc.metadata['title']} Page Number: {doc.metadata['page']}"
            )

        log.info(f"Result: {result['result']}")
    except Exception as e:
        log.error(f"Error during QA chain execution: {str(e)}")

In [10]:
question = "What is the summary of the episode 65?"
qa_documents_with_filters(db, llm, question)



> Entering new RetrievalQA chain...


  0%|          | 0/1 [00:00<?, ?it/s]

2026-03-17 05:49:21,557 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/d12a770eac7c91c9/embeddings?api-version=2025-03-01-preview "HTTP/1.1 200 OK"


100%|██████████| 1/1 [00:00<00:00,  2.84it/s]


2026-03-17 05:49:23,199 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/da74d08d2f277477/chat/completions?api-version=2025-03-01-preview "HTTP/1.1 200 OK"

> Finished chain.
2026-03-17 05:49:23,201 - __main__ - INFO - Source Documents:
2026-03-17 05:49:23,201 - __main__ - INFO - Title: Microsoft Word - Word - Transcript - Episode 64.docx Page Number: 8
2026-03-17 05:49:23,201 - __main__ - INFO - Title: Microsoft Word - Word - Transcript - Episode 64.docx Page Number: 0
2026-03-17 05:49:23,202 - __main__ - INFO - Title: Microsoft Word - Word - Transcript - Episode 65.docx Page Number: 7
2026-03-17 05:49:23,202 - __main__ - INFO - Title: Microsoft Word - Word - Transcript - Episode 65.docx Page Number: 0
2026-03-17 05:49:23,202 - __main__ - INFO - Title: Microsoft Word - Word - Transcript - Episode 65.docx Page Number: 3
2026-03-17 05:49:23,202 - __main__ - INFO - Result: Episode 65 of the Future of Supply Chain Podca

In [ ]:
def qa_with_hana_self_query(db, llm, query):
    """
    Perform a self-query using HANA and the provided LLM.

    Args:
        db: The database object.
        llm: The language model object.
        query: The query string.
    """
    try:
        metadata_field_info = [
            AttributeInfo(
                name="podcast_title", 
                description="Title of the Podcast", 
                type="string"
            ),
            AttributeInfo(
                name="episode",
                description="Episode number of the Podcast",
                type="integer",
            ),
        ]

        document_content_description = "A collection of podcasts from the SAP podcast."
        hana_translator = HanaTranslator()

        retriever = SelfQueryRetriever.from_llm(
            llm,
            db,
            document_content_description,
            metadata_field_info,
            structured_query_translator=hana_translator,
        )

        docs = retriever.invoke(input=query)
        log.info(f"Running the query: {query}")
        log.info("Source document metadata below\n")
        for doc in docs:
            log.info("-" * 80)
            log.info(
                f"Podcast title: {doc.metadata['podcast_title']}\nEpisode: {doc.metadata['episode']}"
            )

        # Concatenate the content of the retrieved documents
        context = " ".join(doc.page_content for doc in docs)
        system_message = f"You are a helpful assistant. Use the following context to answer the user query:\n{context}"
        human_message = query
        messages = [
            ("system", system_message),
            ("human", human_message),
        ]

        answer = llm.invoke(messages)
        log.info("Answer to the query:")
        log.info(answer.content)
    except Exception as e:
        log.error(f"Error performing query: {str(e)}")


In [9]:
from langchain_hana import HanaTranslator
question = "What is the summary of the episode 65?"
qa_with_hana_self_query(db, llm, question)

2026-03-17 05:48:17,014 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/da74d08d2f277477/chat/completions?api-version=2025-03-01-preview "HTTP/1.1 200 OK"


  0%|          | 0/1 [00:00<?, ?it/s]

2026-03-17 05:48:17,229 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/d12a770eac7c91c9/embeddings?api-version=2025-03-01-preview "HTTP/1.1 200 OK"


100%|██████████| 1/1 [00:00<00:00,  4.68it/s]

2026-03-17 05:48:17,323 - __main__ - INFO - Running the query: What is the summary of the episode 65?
2026-03-17 05:48:17,324 - __main__ - INFO - Source document metadata below

2026-03-17 05:48:17,324 - __main__ - INFO - --------------------------------------------------------------------------------
2026-03-17 05:48:17,325 - __main__ - INFO - Podcast title: The Future of Supply Chain: Episode 65: Grounding Your Supply Chain in Data with Google Cloud’s Paula Natoli
Episode: 65
2026-03-17 05:48:17,325 - __main__ - INFO - --------------------------------------------------------------------------------
2026-03-17 05:48:17,325 - __main__ - INFO - Podcast title: The Future of Supply Chain: Episode 65: Grounding Your Supply Chain in Data with Google Cloud’s Paula Natoli
Episode: 65
2026-03-17 05:48:17,325 - __main__ - INFO - --------------------------------------------------------------------------------
2026-03-17 05:48:17,326 - __main__ - INFO - Podcast title: The Future of Supply Chain: 

2026-03-17 05:48:19,150 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/da74d08d2f277477/chat/completions?api-version=2025-03-01-preview "HTTP/1.1 200 OK"
2026-03-17 05:48:19,151 - __main__ - INFO - Answer to the query:
2026-03-17 05:48:19,152 - __main__ - INFO - Episode 65 of the podcast discusses the future of supply chain, focusing on the integration of AI and Generative AI to enhance prediction modeling and efficiency. The conversation highlights the importance of use case-driven technology adoption, rather than technology for its own sake. The speakers emphasize the need for improved visibility and data sharing across supply chain ecosystems, including upstream and downstream partners, to drive business outcomes. They explore how AI can be applied to demand forecasting, delivery predictions, and supplier contracting, among other areas. The episode concludes with the idea that the future of supply chain will be 

In [16]:
def execute_hana_self_query(question):


    print("=" * 10, "User query without database filtering", "=" * 10)
    qa_documents_with_filters(db, llm, question)

    print("=" * 10, "Performing HANA Self query", "=" * 10)
    qa_with_hana_self_query(db, llm, question)

    log.info(
        "Self querying completed successfully!"  
    )

In [17]:
 

question = "What is the summary of the episode 65?"
execute_hana_self_query(question)

========== User query without database filtering ==========


> Entering new RetrievalQA chain...


  0%|          | 0/1 [00:00<?, ?it/s]

2026-03-17 05:53:35,156 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/d12a770eac7c91c9/embeddings?api-version=2025-03-01-preview "HTTP/1.1 200 OK"


100%|██████████| 1/1 [00:00<00:00,  4.46it/s]


2026-03-17 05:53:36,722 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/da74d08d2f277477/chat/completions?api-version=2025-03-01-preview "HTTP/1.1 200 OK"

> Finished chain.
2026-03-17 05:53:36,723 - __main__ - INFO - Source Documents:
2026-03-17 05:53:36,723 - __main__ - INFO - Title: Microsoft Word - Word - Transcript - Episode 64.docx Page Number: 8
2026-03-17 05:53:36,724 - __main__ - INFO - Title: Microsoft Word - Word - Transcript - Episode 64.docx Page Number: 0
2026-03-17 05:53:36,724 - __main__ - INFO - Title: Microsoft Word - Word - Transcript - Episode 65.docx Page Number: 7
2026-03-17 05:53:36,724 - __main__ - INFO - Title: Microsoft Word - Word - Transcript - Episode 65.docx Page Number: 0
2026-03-17 05:53:36,725 - __main__ - INFO - Title: Microsoft Word - Word - Transcript - Episode 65.docx Page Number: 3
2026-03-17 05:53:36,725 - __main__ - INFO - Result: Episode 65 of the "Future of Supply Chain" pod

  0%|          | 0/1 [00:00<?, ?it/s]

2026-03-17 05:53:37,828 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/d12a770eac7c91c9/embeddings?api-version=2025-03-01-preview "HTTP/1.1 200 OK"


100%|██████████| 1/1 [00:00<00:00,  4.76it/s]

2026-03-17 05:53:37,848 - __main__ - INFO - Running the query: What is the summary of the episode 65?
2026-03-17 05:53:37,849 - __main__ - INFO - Source document metadata below

2026-03-17 05:53:37,849 - __main__ - INFO - --------------------------------------------------------------------------------
2026-03-17 05:53:37,849 - __main__ - INFO - Podcast title: The Future of Supply Chain: Episode 65: Grounding Your Supply Chain in Data with Google Cloud’s Paula Natoli
Episode: 65
2026-03-17 05:53:37,849 - __main__ - INFO - --------------------------------------------------------------------------------
2026-03-17 05:53:37,850 - __main__ - INFO - Podcast title: The Future of Supply Chain: Episode 65: Grounding Your Supply Chain in Data with Google Cloud’s Paula Natoli
Episode: 65
2026-03-17 05:53:37,850 - __main__ - INFO - --------------------------------------------------------------------------------
2026-03-17 05:53:37,850 - __main__ - INFO - Podcast title: The Future of Supply Chain: 

2026-03-17 05:53:40,516 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/da74d08d2f277477/chat/completions?api-version=2025-03-01-preview "HTTP/1.1 200 OK"
2026-03-17 05:53:40,517 - __main__ - INFO - Answer to the query:
2026-03-17 05:53:40,517 - __main__ - INFO - Episode 65 of the podcast discusses the future of supply chain, focusing on the integration of AI and Generative AI to enhance prediction modeling and efficiency. The conversation highlights the importance of use case-driven approaches rather than technology for technology's sake. The speakers emphasize the need for improved visibility and data sharing across supply chain ecosystems, including upstream and downstream partners, to drive business outcomes. They explore how AI can be applied to demand forecasting, delivery predictions, and supplier contracting, showcasing examples like using Generative AI to analyze contracts and assess shipment damage. The ep

In [ ]:
 
question = "What is the summary of the episode 67?"
execute_hana_self_query(question)

========== User query without database filtering ==========


> Entering new RetrievalQA chain...


  0%|          | 0/1 [00:00<?, ?it/s]

2026-03-17 05:53:47,961 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/d12a770eac7c91c9/embeddings?api-version=2025-03-01-preview "HTTP/1.1 200 OK"


100%|██████████| 1/1 [00:00<00:00,  4.23it/s]


2026-03-17 05:53:48,916 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/da74d08d2f277477/chat/completions?api-version=2025-03-01-preview "HTTP/1.1 200 OK"

> Finished chain.
2026-03-17 05:53:48,918 - __main__ - INFO - Source Documents:
2026-03-17 05:53:48,918 - __main__ - INFO - Title: Microsoft Word - Word - Transcript - Episode 64.docx Page Number: 8
2026-03-17 05:53:48,919 - __main__ - INFO - Title: Microsoft Word - Word - Transcript - Episode 64.docx Page Number: 0
2026-03-17 05:53:48,919 - __main__ - INFO - Title: Microsoft Word - Word - Transcript - Episode 65.docx Page Number: 7
2026-03-17 05:53:48,919 - __main__ - INFO - Title: Microsoft Word - Word - Transcript - Episode 65.docx Page Number: 3
2026-03-17 05:53:48,919 - __main__ - INFO - Title: Microsoft Word - Word - Transcript - Episode 65.docx Page Number: 0
2026-03-17 05:53:48,920 - __main__ - INFO - Result: I'm sorry, but the provided context does not i

  0%|          | 0/1 [00:00<?, ?it/s]

2026-03-17 05:53:49,915 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/d12a770eac7c91c9/embeddings?api-version=2025-03-01-preview "HTTP/1.1 200 OK"


100%|██████████| 1/1 [00:00<00:00,  4.79it/s]

2026-03-17 05:53:49,927 - __main__ - INFO - Running the query: What is the summary of the episode 67?
2026-03-17 05:53:49,928 - __main__ - INFO - Source document metadata below



2026-03-17 05:53:50,653 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/da74d08d2f277477/chat/completions?api-version=2025-03-01-preview "HTTP/1.1 200 OK"
2026-03-17 05:53:50,654 - __main__ - INFO - Answer to the query:
2026-03-17 05:53:50,654 - __main__ - INFO - To provide a summary of episode 67, I would need to know the name of the show or series you are referring to. Could you please specify that?
2026-03-17 05:53:50,655 - __main__ - INFO - Self querying completed successfully!


## Advanced RAG: Fusion

### Prepare Document

#### Document from git repository

In [5]:
GIT_URL="https://github.com/SAP-docs/btp-cloud-platform"
GITREPO_PATH="./Gitdown/btp-cloud-platform/"
GIT_TABLE_NAME="GIT_DOCS"

##### Load the document 

In [68]:
from langchain_community.document_loaders import GitLoader

# Define a function to fetch file from a git respository
def fetch_gitrepository_docs(gitrepository_url):
    try:
        log.info("Getting the documents from the GitHub repository: %s", gitrepository_url)
        loader = GitLoader(
            clone_url=gitrepository_url,
            repo_path=GITREPO_PATH,
            #file_filter=lambda file_path: file_path.endswith(".md"),
            #file_filter=lambda file_path: file_path.startswith("./Gitdown/btp-cloud-platform/docs")
            file_filter=lambda file_path: file_path.startswith(GITREPO_PATH+"docs/")
            and file_path.endswith(".md"),
            branch="main",    
        )
        documents = loader.load()
        log.info("Documents loaded successfully. count=%d", len(documents) if documents else 0)
        return documents
    except Exception as e:
        log.error(f"Error occurred while loading documents: {str(e)}")

##### Chunk the document 

In [ ]:
# from langchain.text_splitter import RecursiveCharacterTextSplitter  # deprecated
# from langchain.schema import Document                               # deprecated

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# Define a function to chunk fetched document
def split_docs_into_chunks(
    documents: list[Document], chunk_size: int = 1000, chunk_overlap: int = 100
):
    """
    Splits a list of documents into chunks of specified size with overlap.

    Args:
        documents (list[Document]): The list of documents to be split into chunks.
        chunk_size (int, optional): The size of each chunk. Defaults to 1000.
        chunk_overlap (int, optional): The overlap between consecutive chunks. Defaults to 100.

    Returns:
        list[list[Document]]: A list of chunks, where each chunk is a list of documents.

    """
    try:
        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            length_function=len,
            add_start_index=True,
        )
        chunks = text_splitter.split_documents(documents)
        log.info(f"Split {len(documents)} documents into {len(chunks)} chunks.")

        return chunks
    except Exception as e:
        log.error(f"An error occurred while splitting documents into chunks: {str(e)}")
        raise

In [70]:
# Test the function
git_docs=fetch_gitrepository_docs(gitrepository_url=GIT_URL)
chunks=split_docs_into_chunks(git_docs)

2026-03-17 07:57:08,373 - __main__ - INFO - Getting the documents from the GitHub repository: https://github.com/SAP-docs/btp-cloud-platform
2026-03-17 07:57:13,322 - __main__ - INFO - Documents loaded successfully. count=1960
2026-03-17 07:57:13,528 - __main__ - INFO - Split 1960 documents into 11176 chunks.


##### Embedding the document 

In [74]:
# Define a function to do the embedding and then save the data into HANA table
def ingest_git(gitrepository_url):
 
    #log.info(f"Start fetching data from {gitrepository_url}")
    git_docs=fetch_gitrepository_docs(gitrepository_url=gitrepository_url)
    chunks=split_docs_into_chunks(git_docs)
    if not chunks:
        log.warning("No document chunks to ingest.")
        return

    try:
        teardown_hana_table(table_name=GIT_TABLE_NAME)
        log.info(f"Start ingesting data into {GIT_TABLE_NAME}")
        
        db = HanaDB(
            embedding=embeddings, 
            connection= init_env.get_connection_to_hana_db(), 
            table_name=GIT_TABLE_NAME
        )
        db.add_documents(chunks)
        log.success("Added SAP btp docs successfully!")
    except Exception as e:
        log.error(f"Error during SAP documents ingestion: {str(e)}")


In [ ]:
# Run the ingest
ingest_git(gitrepository_url=GIT_URL)

### RAG Fusion

In [32]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser
from langchain.load import dumps, loads
from langchain_hana import HanaDB 

template = """
        Answer the question in detail and as truthfully as possible based only on the provided context. If you're unsure of the question or answer, say "Sorry, I don't know".
        <context>
        {context}
        </context>

        Question: {original_query}
"""
prompt = ChatPromptTemplate.from_template(template)

GIT_TABLE_NAME="GIT_DOCS"
db = HanaDB(
        embedding=embeddings, 
        connection= init_env.get_connection_to_hana_db(), 
        table_name=GIT_TABLE_NAME
)
retriever = db.as_retriever(search_kwargs={"k": 5})



#### Question answering without RAG Fusion

##### Understand Runnable

In [ ]:
# Test the retrived context from Runnable

# 把 dict 包装成一个真正的 Runnable
pre_prompt_stage = RunnableParallel(
    context=retriever,                 # 会把输入字符串当作查询，返回 List[Document]
    original_query=RunnablePassthrough()  # 原样透传输入字符串
)

original_query = "Cloud Foundry, Kyma or what?"

intermediate = pre_prompt_stage.invoke(original_query)

list=(intermediate["context"])              
i=1
for l in list:
    print("-"*150,i,"-"*150)
    print(l.metadata["file_name"])
    print(l.page_content)
    i=i+1

  0%|          | 0/1 [00:00<?, ?it/s]

2026-03-18 05:48:33,791 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/d12a770eac7c91c9/embeddings?api-version=2025-03-01-preview "HTTP/1.1 200 OK"


100%|██████████| 1/1 [00:00<00:00,  2.96it/s]

------------------------------------------------------------------------------------------------------------------------------------------------------ 1 ------------------------------------------------------------------------------------------------------------------------------------------------------
kyma-environment-468c2f3.md
Kyma implements a dedicated application runtime to deploy highly scalable, robust, and secure containerized microservices.

> ### Note:  
> Kyma as a managed service automatically checks all Kyma-managed resources. Any unexpected modifications are discarded, and the resource is reverted to the original state.

Every Kyma environment consists of:

-   One or more Kubernetes clusters based on project "Gardener" on a cloud provider and region \(data center\) of your choice. To find out the available regions and providers, see [Regions for the Kyma Environment](regions-for-the-kyma-environment-557ec3a.md).
-   A set of Kyma modules picked by a user in the release 

In [97]:
# Test the prompt

pre_prompt_stage = {
    "context": retriever,
    "original_query": RunnablePassthrough()
}
 
chain = pre_prompt_stage | prompt 
 
intermediate = chain.invoke(original_query)
print(intermediate.messages)  # 同方式一打印
 

  0%|          | 0/1 [00:00<?, ?it/s]

2026-03-18 07:15:59,867 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/d12a770eac7c91c9/embeddings?api-version=2025-03-01-preview "HTTP/1.1 200 OK"


100%|██████████| 1/1 [00:00<00:00,  3.98it/s]


[HumanMessage(content='\n        Answer the question in detail and as truthfully as possible based only on the provided context. If you\'re unsure of the question or answer, say "Sorry, I don\'t know".\n        <context>\n        [Document(metadata={\'source\': \'docs/10-concepts/kyma-environment-468c2f3.md\', \'file_path\': \'docs/10-concepts/kyma-environment-468c2f3.md\', \'file_name\': \'kyma-environment-468c2f3.md\', \'file_type\': \'.md\', \'start_index\': 944}, page_content=\'Kyma implements a dedicated application runtime to deploy highly scalable, robust, and secure containerized microservices.\\n\\n> ### Note:  \\n> Kyma as a managed service automatically checks all Kyma-managed resources. Any unexpected modifications are discarded, and the resource is reverted to the original state.\\n\\nEvery Kyma environment consists of:\\n\\n-   One or more Kubernetes clusters based on project "Gardener" on a cloud provider and region \\\\(data center\\\\) of your choice. To find out the a

##### Build the function using Runnable

In [77]:
# Question answering without RAG Fusion
def qa_without_fusion(llm, original_query, prompt, retriever):
    chain = (
        {"context": retriever, "original_query": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )
     


    result = chain.invoke(original_query)
    print("-"*300)
    print("Query before rewrite:\n ", original_query)
    print("QA result without query rewrite:\n", result)
    
    return result


In [78]:
# Test the function with different retrived numbers
retriever = db.as_retriever(search_kwargs={"k": 40})
original_query="Cloud Foundry, Kyma or what?"

qa_without_fusion(llm, original_query, prompt, retriever)

  0%|          | 0/1 [00:00<?, ?it/s]

2026-03-18 06:22:16,039 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/d12a770eac7c91c9/embeddings?api-version=2025-03-01-preview "HTTP/1.1 200 OK"


100%|██████████| 1/1 [00:00<00:00,  4.23it/s]


2026-03-18 06:22:19,364 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/da74d08d2f277477/chat/completions?api-version=2025-03-01-preview "HTTP/1.1 200 OK"
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Query before rewrite:
  Cloud Foundry, Kyma or what?
QA result without query rewrite:
 The choice between Cloud Foundry and Kyma environments depends on your specific needs and use cases. Both environments offer distinct advantages and cater to different development and operational requirements:

1. **Cloud Foundry Environment**:
   - **Polyglot Development**: Supports multiple programming languages such as Java, Node.js, Python, and others through community buildpacks

'The choice between Cloud Foundry and Kyma environments depends on your specific needs and use cases. Both environments offer distinct advantages and cater to different development and operational requirements:\n\n1. **Cloud Foundry Environment**:\n   - **Polyglot Development**: Supports multiple programming languages such as Java, Node.js, Python, and others through community buildpacks.\n   - **Application Lifecycle Management**: Provides tools for starting, stopping, scaling, and configuring distributed cloud applications.\n   - **Integration with SAP HANA**: Offers integration with SAP HANA extended application services.\n   - **Open Standards**: Allows building applications on open standards with various buildpacks.\n   - **Multitarget Applications (MTA)**: Supports the development and deployment of applications in the multitarget format.\n\n2. **Kyma Environment**:\n   - **Kubernetes-Based**: Provides a cloud-native Kubernetes application runtime based on the open-source project 

In [81]:
# Compare with simple qa retriver
original_query="Cloud Foundry, Kyma or what?"

# Create the QA instance to query llm based on custom documents
from langchain.chains import RetrievalQA
qa = RetrievalQA.from_llm(
    llm=llm, 
    retriever=retriever, 
    return_source_documents=True
)
answer = qa.invoke(original_query)
print("-"*300)
print("Query before rewrite:\n ", original_query)
print("QA result without query rewrite: ")
print(answer["result"])

  0%|          | 0/1 [00:00<?, ?it/s]

2026-03-18 06:30:59,349 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/d12a770eac7c91c9/embeddings?api-version=2025-03-01-preview "HTTP/1.1 200 OK"


100%|██████████| 1/1 [00:00<00:00,  4.50it/s]


2026-03-18 06:31:03,316 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/da74d08d2f277477/chat/completions?api-version=2025-03-01-preview "HTTP/1.1 200 OK"
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Query before rewrite:
  Cloud Foundry, Kyma or what?
QA result without query rewrite: 
Choosing between Cloud Foundry and Kyma depends on your specific needs and use case scenarios. Here's a brief comparison to help you decide:

1. **Cloud Foundry Environment**:
   - **Purpose**: Ideal for developing polyglot cloud applications.
   - **Languages**: Supports multiple runtimes and programming languages, including Java, Node.js, and Python.
   - **Features**: Offers lifec

In [ ]:
# Compare with answer without RAG
answer = llm.invoke(original_query)
print(answer.content)

2026-03-18 06:29:10,748 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/da74d08d2f277477/chat/completions?api-version=2025-03-01-preview "HTTP/1.1 200 OK"
Choosing between Cloud Foundry, Kyma, or another platform depends on your specific needs, goals, and existing infrastructure. Here's a brief overview of each to help you decide:

### Cloud Foundry
- **Purpose**: Cloud Foundry is an open-source platform as a service (PaaS) that provides a highly efficient, modern model for cloud-native application deployment and management.
- **Strengths**:
  - **Multi-Cloud Support**: Works across various cloud providers, offering flexibility.
  - **Developer Productivity**: Simplifies the deployment process, allowing developers to focus on writing code.
  - **Scalability**: Automatically scales applications based on demand.
  - **Rich Ecosystem**: Supports multiple languages and frameworks.
- **Use Cases**: Ideal for organization

#### Question answering with RAG Fusion

In [5]:

fusion_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant that generates multiple search queries based on a single input query.",
        ),
        ("user", "Generate multiple search queries related to: {original_query}"),
        ("user", "OUTPUT (4 queries):"),
    ]
)

original_query = "Cloud Foundry, Kyma or what?"

from rich.console import Console
from rich.pretty import Pretty
console = Console(soft_wrap=True)  # 换行控制

##### Understand the runnable pipeline

###### Check the prompt template for split

In [ ]:
# check the prompt template for split

pipeline=( 
    RunnablePassthrough()
    |fusion_prompt
    #| llm
    #| StrOutputParser()
    #| print_and_split
    #| retriever.map()
    #| reciprocal_rank_fusion
)
intermediate = pipeline.invoke(original_query)
print(intermediate)
console.print(Pretty(intermediate, indent_guides=True))

messages=[SystemMessage(content='You are a helpful assistant that generates multiple search queries based on a single input query.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Generate multiple search queries related to: Cloud Foundry, Kyma or what?', additional_kwargs={}, response_metadata={}), HumanMessage(content='OUTPUT (4 queries):', additional_kwargs={}, response_metadata={})]


ChatPromptValue(
│   messages=[
│   │   SystemMessage(
│   │   │   content='You are a helpful assistant that generates multiple search queries based on a single input query.',
│   │   │   additional_kwargs={},
│   │   │   response_metadata={}
│   │   ),
│   │   HumanMessage(
│   │   │   content='Generate multiple search queries related to: Cloud Foundry, Kyma or what?',
│   │   │   additional_kwargs={},
│   │   │   response_metadata={}
│   │   ),
│   │   HumanMessage(content='OUTPUT (4 queries):', additional_kwargs={}, response_metadata={})
│   ]
)

###### LLM rewrite prompt

In [ ]:
# Test the rewrite the original query into mutiple questions from Runnable

pipeline=( 
    RunnablePassthrough()
    |fusion_prompt
    | llm
    | StrOutputParser()
    #| print_and_split
    #| retriever.map()
    #| reciprocal_rank_fusion
)
intermediate = pipeline.invoke(original_query)
 
console.print(Pretty(intermediate, indent_guides=True))


2026-03-20 03:16:03,400 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/da74d08d2f277477/chat/completions?api-version=2025-03-01-preview "HTTP/1.1 200 OK"


'1. "Cloud Foundry vs Kyma: Comparing cloud platforms"\n2. "Introduction to Cloud Foundry and Kyma: Key features and benefits"\n3. "Cloud Foundry and Kyma integration: Best practices and tools"\n4. "Alternatives to Cloud Foundry and Kyma for cloud-native development"'

###### Check the split questions

In [6]:
# Function to split the questions into a list
def print_and_split(generated_queries: str):
    print("Generated queries: \n", generated_queries)
    return generated_queries.split("\n")

In [94]:
# To proceed the retrival, change the str of quesions into a list

pipeline=( 
    RunnablePassthrough()
    |fusion_prompt
    | llm
    | StrOutputParser()
    | print_and_split
    #| retriever.map()
    #| reciprocal_rank_fusion
)
intermediate = pipeline.invoke(original_query)

console.print(Pretty(intermediate, indent_guides=True))


2026-03-20 07:32:41,786 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/da74d08d2f277477/chat/completions?api-version=2025-03-01-preview "HTTP/1.1 200 OK"
Generated queries: 
 1. "Cloud Foundry vs Kyma: Which platform to choose for cloud-native applications?"
2. "Comparing Cloud Foundry and Kyma: Features, benefits, and use cases"
3. "Cloud Foundry and Kyma integration: How to use both for cloud development"
4. "Kyma vs Cloud Foundry: Pros and cons for enterprise cloud solutions"


[
│   '1. "Cloud Foundry vs Kyma: Which platform to choose for cloud-native applications?"',
│   '2. "Comparing Cloud Foundry and Kyma: Features, benefits, and use cases"',
│   '3. "Cloud Foundry and Kyma integration: How to use both for cloud development"',
│   '4. "Kyma vs Cloud Foundry: Pros and cons for enterprise cloud solutions"'
]

###### Retrive from these split questions

In [95]:
# Retrive from these split questions

pipeline=( 
    RunnablePassthrough()
    |fusion_prompt
    | llm
    | StrOutputParser()
    | print_and_split
    | retriever.map()
    #| reciprocal_rank_fusion
)
intermediate = pipeline.invoke(original_query)



2026-03-20 07:32:48,975 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/da74d08d2f277477/chat/completions?api-version=2025-03-01-preview "HTTP/1.1 200 OK"
Generated queries: 
 1. "Cloud Foundry vs Kyma: Which platform to choose for cloud-native applications?"
2. "Comparing Cloud Foundry and Kyma: Features, benefits, and use cases"
3. "Cloud Foundry and Kyma integration: How to use both for cloud development"
4. "Kyma or Cloud Foundry: Pros and cons for enterprise cloud solutions"


  0%|          | 0/1 [00:00<?, ?it/s]




2026-03-20 07:32:49,199 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/d12a770eac7c91c9/embeddings?api-version=2025-03-01-preview "HTTP/1.1 200 OK"




100%|██████████| 1/1 [00:00<00:00,  4.70it/s]

2026-03-20 07:32:49,234 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/d12a770eac7c91c9/embeddings?api-version=2025-03-01-preview "HTTP/1.1 200 OK"




100%|██████████| 1/1 [00:00<00:00,  4.00it/s]

2026-03-20 07:32:49,259 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/d12a770eac7c91c9/embeddings?api-version=2025-03-01-preview "HTTP/1.1 200 OK"
2026-03-20 07:32:49,302 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/d12a770eac7c91c9/embeddings?api-version=2025-03-01-preview "HTTP/1.1 200 OK"



100%|██████████| 1/1 [00:00<00:00,  2.96it/s]


In [97]:
# For each questions in the list, 20 answers are retrived based on retriver setting
print(len(intermediate))
print(len(intermediate[0]))
print(len(intermediate[1]))
print(len(intermediate[2]))
print(len(intermediate[3]))

4
30
30
30
30


###### Combine the list-RRF（Reciprocal Rank Fusion）排序融合

In [98]:
# Define the reciprocal rank fusion function that combines the results from multiple retrievers into a single ranked list
def reciprocal_rank_fusion(results: list[list], k=60):
    fused_scores = {}
    for docs in results:
        for rank, doc in enumerate(docs):
            doc_str = dumps(doc)
            fused_scores.setdefault(doc_str, 0)
            fused_scores[doc_str] += 1 / (rank + k)

    reranked_results = [
        (loads(doc), score)
        for doc, score in sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
    ]
    return reranked_results

In [99]:
pipeline=( 
    RunnablePassthrough()
    |fusion_prompt
    | llm
    | StrOutputParser()
    | print_and_split
    | retriever.map()
    | reciprocal_rank_fusion 
)
intermediate = pipeline.invoke(original_query)

2026-03-20 07:33:29,047 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/da74d08d2f277477/chat/completions?api-version=2025-03-01-preview "HTTP/1.1 200 OK"
Generated queries: 
 1. "Cloud Foundry vs Kyma: Which platform to choose for cloud-native applications?"
2. "Comparing Cloud Foundry and Kyma: Features, benefits, and use cases"
3. "Cloud Foundry and Kyma integration: How to use both for cloud development"
4. "Kyma overview: Is it a better alternative to Cloud Foundry for microservices?"


  0%|          | 0/1 [00:00<?, ?it/s]




2026-03-20 07:33:29,279 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/d12a770eac7c91c9/embeddings?api-version=2025-03-01-preview "HTTP/1.1 200 OK"


2026-03-20 07:33:29,282 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/d12a770eac7c91c9/embeddings?api-version=2025-03-01-preview "HTTP/1.1 200 OK"


100%|██████████| 1/1 [00:00<00:00,  4.37it/s]

100%|██████████| 1/1 [00:00<00:00,  4.35it/s]

2026-03-20 07:33:29,381 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/d12a770eac7c91c9/embeddings?api-version=2025-03-01-preview "HTTP/1.1 200 OK"
2026-03-20 07:33:29,381 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/d12a770eac7c91c9/embeddings?api-version=2025-03-01-preview "HTTP/1.1 200 OK"



100%|██████████| 1/1 [00:00<00:00,  3.01it/s]


100%|██████████| 1/1 [00:00<00:00,  2.67it/s]


In [100]:
# The number of search result is reduced after fusion
print(len(intermediate))
print(len(intermediate[0]))

56
2


In [ ]:
# View the RRF result
console.print(Pretty(intermediate, indent_guides=True))

###### Change the combined list into a diction 

In [102]:
pipeline=( 
    RunnablePassthrough()
    |fusion_prompt
    | llm
    | StrOutputParser()
    | print_and_split
    | retriever.map()
    | reciprocal_rank_fusion 
)

pre_prompt_stage = RunnableParallel(
    {
        "context": {
            "question_to_rewrite": pipeline  # 直接复用
        },
        "original_query": RunnablePassthrough(),  # 原样返回
    }
)

intermediate = pre_prompt_stage.invoke(original_query)

2026-03-20 07:33:56,248 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/da74d08d2f277477/chat/completions?api-version=2025-03-01-preview "HTTP/1.1 200 OK"
Generated queries: 
 1. "Cloud Foundry vs Kyma: Comparing cloud platform features and benefits"
2. "Introduction to Cloud Foundry: Key functionalities and use cases"
3. "Kyma platform overview: How it integrates with Kubernetes"
4. "Choosing between Cloud Foundry and Kyma for enterprise applications"



  0%|          | 0/1 [00:00<?, ?it/s]



2026-03-20 07:33:56,471 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/d12a770eac7c91c9/embeddings?api-version=2025-03-01-preview "HTTP/1.1 200 OK"



100%|██████████| 1/1 [00:00<00:00,  4.55it/s]

2026-03-20 07:33:56,516 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/d12a770eac7c91c9/embeddings?api-version=2025-03-01-preview "HTTP/1.1 200 OK"



100%|██████████| 1/1 [00:00<00:00,  3.79it/s]

2026-03-20 07:33:56,528 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/d12a770eac7c91c9/embeddings?api-version=2025-03-01-preview "HTTP/1.1 200 OK"



100%|██████████| 1/1 [00:00<00:00,  3.64it/s]

2026-03-20 07:33:56,581 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/d12a770eac7c91c9/embeddings?api-version=2025-03-01-preview "HTTP/1.1 200 OK"





100%|██████████| 1/1 [00:00<00:00,  2.85it/s]


In [103]:
# Check the output
print(type(intermediate))  
print(len(intermediate)) # Because only 2 fileds in the dictionary
print(len(intermediate['context']['question_to_rewrite']))
print((intermediate['original_query']))

<class 'dict'>
2
88
Cloud Foundry, Kyma or what?


In [ ]:
# View the full result
console.print(Pretty(intermediate, indent_guides=True))

###### Generate the final prompt

In [133]:
template = """
    Answer the question in detail and as truthfully as possible based only on the provided context. If you're unsure of the question or answer, say "Sorry, I don't know".
    <context>
    {context}
    </context>

    Question: {original_query}
"""

final_prompt = ChatPromptTemplate.from_template(template)
 

chain = (
    pre_prompt_stage
    | final_prompt
    #| llm
    #| StrOutputParser()
)

intermediate = chain.invoke(original_query)

2026-03-20 07:40:52,071 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/da74d08d2f277477/chat/completions?api-version=2025-03-01-preview "HTTP/1.1 200 OK"
Generated queries: 
 1. "Cloud Foundry vs Kyma: Which platform to choose for cloud-native applications?"
2. "Comparing Cloud Foundry and Kyma: Features, benefits, and use cases"
3. "Cloud Foundry and Kyma integration: How to use both for cloud development"
4. "Kyma overview: Is it a better alternative to Cloud Foundry for microservices?"


  0%|          | 0/1 [00:00<?, ?it/s]




2026-03-20 07:40:52,302 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/d12a770eac7c91c9/embeddings?api-version=2025-03-01-preview "HTTP/1.1 200 OK"


100%|██████████| 1/1 [00:00<00:00,  4.37it/s]

2026-03-20 07:40:52,310 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/d12a770eac7c91c9/embeddings?api-version=2025-03-01-preview "HTTP/1.1 200 OK"





100%|██████████| 1/1 [00:00<00:00,  4.37it/s]

2026-03-20 07:40:52,315 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/d12a770eac7c91c9/embeddings?api-version=2025-03-01-preview "HTTP/1.1 200 OK"




100%|██████████| 1/1 [00:00<00:00,  4.26it/s]


2026-03-20 07:40:52,883 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/d12a770eac7c91c9/embeddings?api-version=2025-03-01-preview "HTTP/1.1 200 OK"


100%|██████████| 1/1 [00:00<00:00,  1.24it/s]


In [134]:
# View the full result
console.print(Pretty(intermediate, indent_guides=True))

ChatPromptValue(
│   messages=[
│   │   HumanMessage(
│   │   │   content='\n    Answer the question in detail and as truthfully as possible based only on the provided context. If you\'re unsure of the question or answer, say "Sorry, I don\'t know".\n    <context>\n    {\'question_to_rewrite\': [(Document(metadata={\'source\': \'docs/10-concepts/kyma-environment-468c2f3.md\', \'file_path\': \'docs/10-concepts/kyma-environment-468c2f3.md\', \'file_name\': \'kyma-environment-468c2f3.md\', \'file_type\': \'.md\', \'start_index\': 0}, page_content=\'<!-- loio468c2f3c3ca24c2c8497ef9f83154c44 -->\\n\\n# Kyma Environment\\n\\nSAP BTP, Kyma runtime provides a fully managed cloud-native Kubernetes application runtime based on the open-source project "Kyma". Based on modular building blocks, Kyma runtime includes all the necessary capabilities to simplify the development and to run enterprise-grade cloud-native applications.\\n\\n\\n\\n<a name="loio468c2f3c3ca24c2c8497ef9f83154c44__section_lx1_yxp_nrb"/>\\n\\n## Kyma as a Managed Service\\n\\nKyma environment permits a native consumption of the Multi-Cloud Foundation Services \\\\([What Is the Multi-Cloud Foundation?](https://help.sap.com/viewer/b017fc4f944e4eb5b31501b3d1b6a1f0/Cloud/en-US/06b6fb3d45d040429e36f0359d2fe1f2.html "Get to know the multi-cloud foundation and its environments.") :arrow_upper_right:\\\\) and smooth consumption of SAP and non-SAP applications. It also supports out-of-the-box CAP, SAP Cloud SDK, application router, and HTML5 deployer.\'), 0.06612021857923497), (Document(metadata={\'source\': \'docs/10-concepts/kyma-environment-468c2f3.md\', \'file_path\': \'docs/10-concepts/kyma-environment-468c2f3.md\', \'file_name\': \'kyma-environment-468c2f3.md\', \'file_type\': \'.md\', \'start_index\': 944}, page_content=\'Kyma implements a dedicated application runtime to deploy highly scalable, robust, and secure containerized microservices.\\n\\n> ### Note:  \\n> Kyma as a managed service automatically checks all Kyma-managed resources. Any unexpected modifications are discarded, and the resource is reverted to the original state.\\n\\nEvery Kyma environment consists of:\\n\\n-   One or more Kubernetes clusters based on project "Gardener" on a cloud provider and region \\\\(data center\\\\) of your choice. To find out the available regions and providers, see [Regions for the Kyma Environment](regions-for-the-kyma-environment-557ec3a.md).\\n-   A set of Kyma modules picked by a user in the release channel of their choice installed on the provisioned cluster.\\n\\n![](images/SKR_stack_6b4e9b8.png)\\n\\n\\n\\n<a name="loio468c2f3c3ca24c2c8497ef9f83154c44__section_lyr_gyp_nrb"/>\\n\\n## Integration\'), 0.06223292836196062), (Document(metadata={\'source\': \'docs/40-extensions/extending-sap-solutions-346864d.md\', \'file_path\': \'docs/40-extensions/extending-sap-solutions-346864d.md\', \'file_name\': \'extending-sap-solutions-346864d.md\', \'file_type\': \'.md\', \'start_index\': 6330}, page_content=\'[Extending SAP Marketing Cloud in the Cloud Foundry and Kyma Environment](extending-sap-marketing-cloud-in-the-cloud-foundry-and-kyma-environment-18bb3d9.md "")\'), 0.05978886083052749), (Document(metadata={\'source\': \'docs/30-development/development-in-the-kyma-environment-606ec61.md\', \'file_path\': \'docs/30-development/development-in-the-kyma-environment-606ec61.md\', \'file_name\': \'development-in-the-kyma-environment-606ec61.md\', \'file_type\': \'.md\', \'start_index\': 1648}, page_content=\'For those who prefer to work with command-line tools, the Kyma runtime also offers the Kubernetes command-line tool, [kubectl](https://kubernetes.io/docs/tasks/tools/#kubectl).\\n\\n\\n\\n<a name="loio606ec610ee4746c09d5d2bef5a85a124__section_nhd_xcy_bzb"/>\\n\\n## SAP Cloud Application Programming Model\\n\\nThe SAP Cloud Application Programming Model \\\\(CAP\\\\) is the recommended framework for application and service development in the Kyma runtime. To learn more, see [Prog

###### Get the LLM answer

In [139]:
template = """
    Answer the question in detail and as truthfully as possible based only on the provided context. If you're unsure of the question or answer, say "Sorry, I don't know".
    <context>
    {context}
    </context>

    Question: {original_query}
"""

final_prompt = ChatPromptTemplate.from_template(template)
 

chain = (
    pre_prompt_stage
    | final_prompt
    | llm
    #| StrOutputParser()
)

intermediate = chain.invoke(original_query)

2026-03-20 07:41:49,297 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/da74d08d2f277477/chat/completions?api-version=2025-03-01-preview "HTTP/1.1 200 OK"
Generated queries: 
 1. "Cloud Foundry vs Kyma: Key differences and use cases"
2. "Introduction to Cloud Foundry: Features and benefits"
3. "Getting started with Kyma: A beginner's guide"
4. "Cloud Foundry and Kyma: Which platform is right for your business?"


  0%|          | 0/1 [00:00<?, ?it/s]




2026-03-20 07:41:49,518 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/d12a770eac7c91c9/embeddings?api-version=2025-03-01-preview "HTTP/1.1 200 OK"


100%|██████████| 1/1 [00:00<00:00,  4.57it/s]

2026-03-20 07:41:49,584 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/d12a770eac7c91c9/embeddings?api-version=2025-03-01-preview "HTTP/1.1 200 OK"


...

2026-03-20 07:41:49,630 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/d12a770eac7c91c9/embeddings?api-version=2025-03-01-preview "HTTP/1.1 200 OK"




100%|██████████| 1/1 [00:00<00:00,  3.10it/s]


2026-03-20 07:41:49,854 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/d12a770eac7c91c9/embeddings?api-version=2025-03-01-preview "HTTP/1.1 200 OK"


100%|██████████| 1/1 [00:00<00:00,  1.80it/s]


2026-03-20 07:41:54,634 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/da74d08d2f277477/chat/completions?api-version=2025-03-01-preview "HTTP/1.1 200 OK"


In [140]:
# View the full result
console.print(Pretty(intermediate, indent_guides=True))

AIMessage(
│   content='The context provided discusses two environments: Cloud Foundry and Kyma, both of which are part of the SAP Business Technology Platform (SAP BTP). Here\'s a detailed explanation based on the context:\n\n1. **Cloud Foundry Environment**:\n   - It is an open Platform-as-a-Service (PaaS) that allows for the development and orchestration of polyglot cloud applications.\n   - Supports multiple runtimes, programming languages, libraries, and services.\n   - Offers a variety of buildpacks, including community and self-developed ones.\n   - Integrates with SAP HANA extended application services.\n   - Suitable for developing new business applications and services.\n   - Provides tools for managing the lifecycle of applications, optimizing development and operations, and using application programming models.\n\n2. **Kyma Environment**:\n   - Provides a fully managed cloud-native Kubernetes application runtime based on the open-source project "Kyma".\n   - Includes capabilities to simplify the development and running of enterprise-grade cloud-native applications.\n   - Supports the SAP Cloud Application Programming Model (CAP) for application and service development.\n   - Allows for the creation of microservices in any language and running them as containerized applications.\n   - Offers a central administration dashboard for managing microservices and functions.\n   - Supports integration with SAP and non-SAP applications and services.\n\nBoth environments offer unique features and capabilities, and the choice between them depends on the specific requirements of the application or service being developed. If you are looking for a more Kubernetes-native approach with modular capabilities, Kyma might be the right choice. On the other hand, if you prefer a PaaS with a wide range of language support and buildpacks, Cloud Foundry could be more suitable.',
│   additional_kwargs={'refusal': None},
│   response_metadata={
│   │   'token_usage': {
│   │   │   'completion_tokens': 346,
│   │   │   'prompt_tokens': 27724,
│   │   │   'total_tokens': 28070,
│   │   │   'completion_tokens_details': {
│   │   │   │   'accepted_prediction_tokens': 0,
│   │   │   │   'audio_tokens': 0,
│   │   │   │   'reasoning_tokens': 0,
│   │   │   │   'rejected_prediction_tokens': 0
│   │   │   },
│   │   │   'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}
│   │   },
│   │   'model_name': 'gpt-4o-2024-08-06',
│   │   'system_fingerprint': 'fp_e9b9b028d7',
│   │   'id': 'chatcmpl-DLOmMd8y6Xu1pzLs12K8lSEHd7HDV',
│   │   'service_tier': 'default',
│   │   'finish_reason': 'stop',
│   │   'logprobs': None
│   },
│   id='run--019d0a31-8200-7232-ae38-5f964ab5fd74-0',
│   usage_metadata={
│   │   'input_tokens': 27724,
│   │   'output_tokens': 346,
│   │   'total_tokens': 28070,
│   │   'input_token_details': {'audio': 0, 'cache_read': 0},
│   │   'output_token_details': {'audio': 0, 'reasoning': 0}
│   }
)

In [141]:
print(intermediate)

content='The context provided discusses two environments: Cloud Foundry and Kyma, both of which are part of the SAP Business Technology Platform (SAP BTP). Here\'s a detailed explanation based on the context:\n\n1. **Cloud Foundry Environment**:\n   - It is an open Platform-as-a-Service (PaaS) that allows for the development and orchestration of polyglot cloud applications.\n   - Supports multiple runtimes, programming languages, libraries, and services.\n   - Offers a variety of buildpacks, including community and self-developed ones.\n   - Integrates with SAP HANA extended application services.\n   - Suitable for developing new business applications and services.\n   - Provides tools for managing the lifecycle of applications, optimizing development and operations, and using application programming models.\n\n2. **Kyma Environment**:\n   - Provides a fully managed cloud-native Kubernetes application runtime based on the open-source project "Kyma".\n   - Includes capabilities to simpl

###### Generate the final answer

In [147]:
template = """
    Answer the question in detail and as truthfully as possible based only on the provided context. If you're unsure of the question or answer, say "Sorry, I don't know".
    <context>
    {context}
    </context>

    Question: {original_query}
"""

final_prompt = ChatPromptTemplate.from_template(template)
 

chain = (
    pre_prompt_stage
    | final_prompt
    | llm
    | StrOutputParser()
)

intermediate = chain.invoke(original_query)

2026-03-20 07:42:24,883 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/da74d08d2f277477/chat/completions?api-version=2025-03-01-preview "HTTP/1.1 200 OK"
Generated queries: 
 1. "Cloud Foundry vs Kyma: Comparing cloud platform features and benefits"
2. "Introduction to Cloud Foundry: Key components and use cases"
3. "Kyma project overview: How it integrates with Kubernetes"
4. "Choosing between Cloud Foundry and Kyma for enterprise applications"


  0%|          | 0/1 [00:00<?, ?it/s]




2026-03-20 07:42:25,132 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/d12a770eac7c91c9/embeddings?api-version=2025-03-01-preview "HTTP/1.1 200 OK"


100%|██████████| 1/1 [00:00<00:00,  4.05it/s]

2026-03-20 07:42:25,179 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/d12a770eac7c91c9/embeddings?api-version=2025-03-01-preview "HTTP/1.1 200 OK"


...

2026-03-20 07:42:25,298 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/d12a770eac7c91c9/embeddings?api-version=2025-03-01-preview "HTTP/1.1 200 OK"





100%|██████████| 1/1 [00:00<00:00,  2.46it/s]


2026-03-20 07:42:25,879 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/d12a770eac7c91c9/embeddings?api-version=2025-03-01-preview "HTTP/1.1 200 OK"


100%|██████████| 1/1 [00:00<00:00,  1.01it/s]


2026-03-20 07:42:27,105 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/da74d08d2f277477/chat/completions?api-version=2025-03-01-preview "HTTP/1.1 200 OK"


In [149]:
# View the final result
console.print(Pretty(intermediate, indent_guides=True))

"Sorry, I don't know."

##### The full RAG Fusion function

In [114]:
def qa_with_rag_fusion(llm, retriever,query):
    fusion_prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                "You are a helpful assistant that generates multiple search queries based on a single input query.",
            ),
            ("user", "Generate multiple search queries related to: {query}"),
            ("user", "OUTPUT (4 queries):"),
        ]
    )
    
    template = """
        Answer the question in detail and as truthfully as possible based only on the provided context. If you're unsure of the question or answer, say "Sorry, I don't know".
        <context>
        {context}
        </context>

        Question: {original_query}
    """
    original_prompt = ChatPromptTemplate.from_template(template)
 
    
    def print_and_split(generated_queries: str):
        print("Generated queries: \n", generated_queries)
        return generated_queries.split("\n")

    
    print(f"Original question: {query}")
    # Define the pipeline that generates multiple search queries based on a single input query
    pipeline=( 
    RunnablePassthrough()
        |fusion_prompt
        | llm
        | StrOutputParser()
        | print_and_split
        | retriever.map()
        | reciprocal_rank_fusion
    )
    
    
    chain = (
        {
            "context": pipeline ,
            "original_query": RunnablePassthrough(),
        }
        | original_prompt
        | llm
        | StrOutputParser()
    )

    result = chain.invoke(original_query)

    print("Result with RAG Fusion: ", result)

In [90]:
# Optional: change different temperature and search numbers to test
LLM_MODEL_NAME = 'gpt-4o'
proxy_client = get_proxy_client("gen-ai-hub")
rate_limiter = InMemoryRateLimiter(
    requests_per_second=0.5,  # We can only make a request once every 5 seconds
    check_every_n_seconds=0.1,  # Wake up every 100 ms to check whether allowed to make a request,
    max_bucket_size=10,  # Controls the maximum burst size.
)

llm = ChatOpenAI(
    proxy_model_name=LLM_MODEL_NAME,
    proxy_client=proxy_client,
    temperature=0,
    rate_limiter=rate_limiter,
)
retriever = db.as_retriever(search_kwargs={"k": 30})


In [116]:
query="Cloud Foundry, Kyma or what?"
qa_with_rag_fusion(
    llm=llm, 
    retriever=retriever,
    query=query
)

Original question: Cloud Foundry, Kyma or what?


2026-03-20 07:35:41,548 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/da74d08d2f277477/chat/completions?api-version=2025-03-01-preview "HTTP/1.1 200 OK"
Generated queries: 
 1. "Cloud Foundry vs Kyma: Comparing cloud platforms"
2. "Introduction to Cloud Foundry and Kyma: Key features and benefits"
3. "Alternatives to Cloud Foundry and Kyma for cloud-native development"
4. "Cloud Foundry and Kyma integration: Best practices and use cases"


  0%|          | 0/1 [00:00<?, ?it/s]




2026-03-20 07:35:41,788 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/d12a770eac7c91c9/embeddings?api-version=2025-03-01-preview "HTTP/1.1 200 OK"


100%|██████████| 1/1 [00:00<00:00,  4.30it/s]

...

2026-03-20 07:35:41,806 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/d12a770eac7c91c9/embeddings?api-version=2025-03-01-preview "HTTP/1.1 200 OK"



100%|██████████| 1/1 [00:00<00:00,  3.88it/s]

2026-03-20 07:35:41,836 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/d12a770eac7c91c9/embeddings?api-version=2025-03-01-preview "HTTP/1.1 200 OK"




100%|██████████| 1/1 [00:00<00:00,  2.41it/s]


2026-03-20 07:35:47,219 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/da74d08d2f277477/chat/completions?api-version=2025-03-01-preview "HTTP/1.1 200 OK"
Result with RAG Fusion:  The context provided does not directly answer the question "Cloud Foundry, Kyma or what?" However, it does provide some information about both Cloud Foundry and Kyma environments within the SAP Business Technology Platform (BTP).

1. **Kyma Environment**: 
   - Kyma is a fully managed cloud-native Kubernetes application runtime based on the open-source project "Kyma". It simplifies the development and running of enterprise-grade cloud-native applications.
   - It allows for the deployment of highly scalable, robust, and secure containerized microservices.
   - Kyma as a managed service automatically checks all Kyma-managed resources, discarding any unexpected modifications.
   - It supports the native consumption of Multi-Cloud Foundation 

## Advanced RAG: Rewrite

In [31]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser
from langchain.retrievers.self_query.base import SelfQueryRetriever
from langchain_hana import HanaTranslator
from langchain_hana import HanaDB

template = """
    Answer the question in detail and as truthfully as possible based only on the provided context. If you're unsure of the question or answer, say "Sorry, I don't know".
    <context>
    {context}
    </context>

    Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)


GIT_TABLE_NAME="GIT_DOCS"
db = HanaDB(
        embedding=embeddings, 
        connection= init_env.get_connection_to_hana_db(), 
        table_name=GIT_TABLE_NAME
)
 

def retrieve_chunks(query):
    retrieved_chunks = db.similarity_search(query, k=30)
    return retrieved_chunks


simple_query = "Cloud Foundry, Kyma or what?"


In [ ]:
# Test the function
chunks=retrieve_chunks(simple_query)
console.print(Pretty(chunks, indent_guides=True))

### Question answering without rewrite

#### Understand Runnable

In [30]:
 
chain = (
    {"context": retrieve_chunks, "question": RunnablePassthrough()}
    | prompt
    #| llm
    #| StrOutputParser()
)
print("Query before rewrite: \n", simple_query)

intermediate = chain.invoke(simple_query)

console.print(Pretty(intermediate, indent_guides=True))
 

Query before rewrite: 
 Cloud Foundry, Kyma or what?


  0%|          | 0/1 [00:00<?, ?it/s]

2026-03-23 06:47:11,483 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/d12a770eac7c91c9/embeddings?api-version=2025-03-01-preview "HTTP/1.1 200 OK"


100%|██████████| 1/1 [00:00<00:00,  3.51it/s]


ChatPromptValue(
│   messages=[
│   │   HumanMessage(
│   │   │   content='\n    Answer the question in detail and as truthfully as possible based only on the provided context. If you\'re unsure of the question or answer, say "Sorry, I don\'t know".\n    <context>\n    [Document(metadata={\'source\': \'docs/10-concepts/kyma-environment-468c2f3.md\', \'file_path\': \'docs/10-concepts/kyma-environment-468c2f3.md\', \'file_name\': \'kyma-environment-468c2f3.md\', \'file_type\': \'.md\', \'start_index\': 944}, page_content=\'Kyma implements a dedicated application runtime to deploy highly scalable, robust, and secure containerized microservices.\\n\\n> ### Note:  \\n> Kyma as a managed service automatically checks all Kyma-managed resources. Any unexpected modifications are discarded, and the resource is reverted to the original state.\\n\\nEvery Kyma environment consists of:\\n\\n-   One or more Kubernetes clusters based on project "Gardener" on a cloud provider and region \\\\(data center\\\\) of your choice. To find out the available regions and providers, see [Regions for the Kyma Environment](regions-for-the-kyma-environment-557ec3a.md).\\n-   A set of Kyma modules picked by a user in the release channel of their choice installed on the provisioned cluster.\\n\\n![](images/SKR_stack_6b4e9b8.png)\\n\\n\\n\\n<a name="loio468c2f3c3ca24c2c8497ef9f83154c44__section_lyr_gyp_nrb"/>\\n\\n## Integration\'), Document(metadata={\'source\': \'docs/10-concepts/kyma-environment-468c2f3.md\', \'file_path\': \'docs/10-concepts/kyma-environment-468c2f3.md\', \'file_name\': \'kyma-environment-468c2f3.md\', \'file_type\': \'.md\', \'start_index\': 0}, page_content=\'<!-- loio468c2f3c3ca24c2c8497ef9f83154c44 -->\\n\\n# Kyma Environment\\n\\nSAP BTP, Kyma runtime provides a fully managed cloud-native Kubernetes application runtime based on the open-source project "Kyma". Based on modular building blocks, Kyma runtime includes all the necessary capabilities to simplify the development and to run enterprise-grade cloud-native applications.\\n\\n\\n\\n<a name="loio468c2f3c3ca24c2c8497ef9f83154c44__section_lx1_yxp_nrb"/>\\n\\n## Kyma as a Managed Service\\n\\nKyma environment permits a native consumption of the Multi-Cloud Foundation Services \\\\([What Is the Multi-Cloud Foundation?](https://help.sap.com/viewer/b017fc4f944e4eb5b31501b3d1b6a1f0/Cloud/en-US/06b6fb3d45d040429e36f0359d2fe1f2.html "Get to know the multi-cloud foundation and its environments.") :arrow_upper_right:\\\\) and smooth consumption of SAP and non-SAP applications. It also supports out-of-the-box CAP, SAP Cloud SDK, application router, and HTML5 deployer.\'), Document(metadata={\'source\': \'docs/30-development/development-in-the-kyma-environment-606ec61.md\', \'file_path\': \'docs/30-development/development-in-the-kyma-environment-606ec61.md\', \'file_name\': \'development-in-the-kyma-environment-606ec61.md\', \'file_type\': \'.md\', \'start_index\': 1648}, page_content=\'For those who prefer to work with command-line tools, the Kyma runtime also offers the Kubernetes command-line tool, [kubectl](https://kubernetes.io/docs/tasks/tools/#kubectl).\\n\\n\\n\\n<a name="loio606ec610ee4746c09d5d2bef5a85a124__section_nhd_xcy_bzb"/>\\n\\n## SAP Cloud Application Programming Model\\n\\nThe SAP Cloud Application Programming Model \\\\(CAP\\\\) is the recommended framework for application and service development in the Kyma runtime. To learn more, see [Programming Models](../10-concepts/programming-models-042061d.md).\\n\\n\\n\\n<a name="loio606ec610ee4746c09d5d2bef5a85a124__section_zm5_4pl_qxb"/>\\n\\n## Using API\'), Document(metadata={\'source\': \'docs/50-administration-and-ops/adding-and-deleting-a-kyma-module-1b548e9.md\', \'file_path\': \'docs/50-administration-and-ops/adding-and-deleting-a-kyma-module-1b548e9.md\', \'file_name\': \'adding-and-deleting-a-kyma-module-1b548e9.md\', \'file_type\': \'.md\', \'start_index\': 3426}, page_content=\'[Kyma CLI](../10-concepts/kyma-cli-292454

In [181]:
 
chain = (
    {"context": retrieve_chunks, "question": RunnablePassthrough()}
    | prompt
    | llm
    #| StrOutputParser()
)
print("Query before rewrite: \n", simple_query)

intermediate = chain.invoke(simple_query)

console.print(Pretty(intermediate, indent_guides=True))
 

Query before rewrite: 
 Cloud Foundry, Kyma or what?


  0%|          | 0/1 [00:00<?, ?it/s]

2026-03-20 08:38:27,340 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/d12a770eac7c91c9/embeddings?api-version=2025-03-01-preview "HTTP/1.1 200 OK"


100%|██████████| 1/1 [00:00<00:00,  3.36it/s]


2026-03-20 08:38:30,414 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/da74d08d2f277477/chat/completions?api-version=2025-03-01-preview "HTTP/1.1 200 OK"


AIMessage(
│   content='The context provided discusses both Cloud Foundry and Kyma environments as options for developing and deploying applications on SAP Business Technology Platform (BTP). \n\n1. **Cloud Foundry Environment**: \n   - It is an open Platform-as-a-Service (PaaS) targeted at microservice development and orchestration.\n   - Supports polyglot applications, allowing developers to use various programming languages such as Java, Node.js, Python, PHP, Ruby, and Go.\n   - Offers lifecycle management of applications, including starting, stopping, scaling, and configuring distributed cloud applications.\n   - Integrates with SAP HANA extended application services and supports multiple runtimes, programming languages, libraries, and services.\n\n2. **Kyma Environment**:\n   - Provides a fully managed cloud-native Kubernetes application runtime based on the open-source project "Kyma".\n   - Allows deployment of highly scalable, robust, and secure containerized microservices.\n   - Supports modular building blocks to simplify the development and running of enterprise-grade cloud-native applications.\n   - Offers integration with SAP systems, enabling the building of serverless applications called "Functions" that can react to events or API calls.\n   - Supports the SAP Cloud Application Programming Model (CAP) for application and service development.\n\nBoth environments have their unique features and capabilities, and the choice between them depends on the specific requirements and preferences of the development project. If you need more complexity in your account model or want to share SAP BTP services, you might consider setting up subaccounts that run both Cloud Foundry and Kyma. \n\nSorry, I don\'t know if there is another option beyond Cloud Foundry and Kyma based on the provided context.',
│   additional_kwargs={'refusal': None},
│   response_metadata={
│   │   'token_usage': {
│   │   │   'completion_tokens': 337,
│   │   │   'prompt_tokens': 8923,
│   │   │   'total_tokens': 9260,
│   │   │   'completion_tokens_details': {
│   │   │   │   'accepted_prediction_tokens': 0,
│   │   │   │   'audio_tokens': 0,
│   │   │   │   'reasoning_tokens': 0,
│   │   │   │   'rejected_prediction_tokens': 0
│   │   │   },
│   │   │   'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 1280}
│   │   },
│   │   'model_name': 'gpt-4o-2024-08-06',
│   │   'system_fingerprint': 'fp_e9b9b028d7',
│   │   'id': 'chatcmpl-DLPf9CZ0NkmGPrF5O3Lso5ZCHM2vi',
│   │   'service_tier': 'default',
│   │   'finish_reason': 'stop',
│   │   'logprobs': None
│   },
│   id='run--019d0a65-595b-7700-96a2-e6715a2cee93-0',
│   usage_metadata={
│   │   'input_tokens': 8923,
│   │   'output_tokens': 337,
│   │   'total_tokens': 9260,
│   │   'input_token_details': {'audio': 0, 'cache_read': 1280},
│   │   'output_token_details': {'audio': 0, 'reasoning': 0}
│   }
)

#### Full function

In [184]:
def invoke_query_without_rewrite(llm, simple_query, prompt, retriever):
    chain = (
        {"context": retrieve_chunks, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )
    print("Query before rewrite: \n", simple_query)

    result = chain.invoke(simple_query)
    print("QA result without query rewrite:\n ", result)
    return result

In [188]:
result=invoke_query_without_rewrite(llm, simple_query, prompt, retrieve_chunks)

Query before rewrite: 
 Cloud Foundry, Kyma or what?


  0%|          | 0/1 [00:00<?, ?it/s]

2026-03-20 08:56:42,559 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/d12a770eac7c91c9/embeddings?api-version=2025-03-01-preview "HTTP/1.1 200 OK"


100%|██████████| 1/1 [00:00<00:00,  1.00it/s]


2026-03-20 08:56:46,973 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/da74d08d2f277477/chat/completions?api-version=2025-03-01-preview "HTTP/1.1 200 OK"
QA result without query rewrite:
  The context provided discusses both the Cloud Foundry and Kyma environments as options for developing applications on SAP Business Technology Platform (BTP). 

1. **Cloud Foundry Environment**: 
   - It is an open Platform-as-a-Service (PaaS) targeted at microservice development and orchestration.
   - Supports polyglot applications, allowing developers to use languages like SAP Java, Node.js, Python, PHP, Ruby, and Go.
   - Offers lifecycle management of applications, including starting, stopping, scaling, and configuring distributed cloud applications.
   - Integrates with SAP HANA extended application services and supports multiple runtimes, programming languages, libraries, and services.

2. **Kyma Environment**:
   - Provide

### Question answering with rewrite

#### Understand Runnable

In [28]:
rewrite_template = """Provide a better query for
the database similarity retrieval and large language model to answer the given question.
Question: {question_to_rewrite}
Answer:"""

rewrite_prompt = ChatPromptTemplate.from_template(rewrite_template)

rewriter = rewrite_prompt | llm | StrOutputParser()

#Test
console.print(Pretty(rewriter, indent_guides=True))


RunnableSequence(
│   first=ChatPromptTemplate(
│   │   input_variables=['question_to_rewrite'],
│   │   input_types={},
│   │   partial_variables={},
│   │   messages=[
│   │   │   HumanMessagePromptTemplate(
│   │   │   │   prompt=PromptTemplate(
│   │   │   │   │   input_variables=['question_to_rewrite'],
│   │   │   │   │   input_types={},
│   │   │   │   │   partial_variables={},
│   │   │   │   │   template='Provide a better query for\nthe database similarity retrieval and large language model to answer the given question.\nQuestion: {question_to_rewrite}\nAnswer:'
│   │   │   │   ),
│   │   │   │   additional_kwargs={}
│   │   │   )
│   │   ]
│   ),
│   middle=[
│   │   ChatOpenAI(
│   │   │   rate_limiter=<langchain_core.rate_limiters.InMemoryRateLimiter object at 0x7f1cd1896a50>,
│   │   │   client=<gen_ai_hub.proxy.native.openai.clients.ChatCompletions object at 0x7f1cd174af90>,
│   │   │   async_client=<gen_ai_hub.proxy.native.openai.clients.AsyncChatCompletions object at 0x7f1cd1600440>,
│   │   │   root_client=<gen_ai_hub.proxy.native.openai.clients.OpenAI object at 0x7f1cd1897620>,
│   │   │   root_async_client=<gen_ai_hub.proxy.native.openai.clients.AsyncOpenAI object at 0x7f1cd174b770>,
│   │   │   model_name='gpt-4o',
│   │   │   temperature=0.1,
│   │   │   model_kwargs={},
│   │   │   openai_api_key=SecretStr('**********'),
│   │   │   n=1,
│   │   │   proxy_client=GenAIHubProxyClient(
│   │   │   │   base_url=None,
│   │   │   │   auth_url=None,
│   │   │   │   client_id=None,
│   │   │   │   client_secret=None,
│   │   │   │   resource_group=None,
│   │   │   │   ai_core_client=<ai_core_sdk.ai_core_v2_client.AICoreV2Client object at 0x7f1cd18967b0>
│   │   │   ),
│   │   │   deployment_id='da74d08d2f277477',
│   │   │   config_name='islm.dpl.cfg.ZTEST_GENAI_OPENAI.7AB55BB2EC021FD186C6F4447E709D45.Deployed based on model ZGPTOPENAI4 training 1',
│   │   │   config_id='7eb56362-ff64-421e-9f91-cb765773ddf9',
│   │   │   proxy_model_name='gpt-4o'
│   │   )
│   ],
│   last=StrOutputParser()
)

In [21]:
def rewritten_query(rewritten_query: str):
    print("Query after rewrite: \n", rewritten_query)
    return rewritten_query



In [22]:
rewrite_retrieve_read_chain = (
        rewriter
        | {
            "context": {"question_to_rewrite": RunnablePassthrough() | retrieve_chunks},
            "question": RunnablePassthrough() | rewritten_query,
        }
        #| prompt
        #| llm
        #| StrOutputParser()
    )

intermediate = rewrite_retrieve_read_chain.invoke(simple_query)

 

console.print(Pretty(intermediate, indent_guides=True))




2026-03-23 05:41:48,620 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/da74d08d2f277477/chat/completions?api-version=2025-03-01-preview "HTTP/1.1 200 OK"
Query after rewrite: 
 To improve the query for database similarity retrieval and leverage a large language model to answer the question effectively, it's important to provide more context and specify what you're looking to compare or understand about Cloud Foundry and Kyma. Here's a refined query:

---

**Query:**

"I'm evaluating cloud-native platforms and need to understand the differences and use cases for Cloud Foundry and Kyma. Could you provide a detailed comparison of their features, benefits, and ideal scenarios for use? Additionally, are there other platforms I should consider in this space?"

---

This query provides clarity on what information is needed and invites a comprehensive comparison, which will help both the database retrieval system and the l

  0%|          | 0/1 [00:00<?, ?it/s]

2026-03-23 05:41:48,871 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/d12a770eac7c91c9/embeddings?api-version=2025-03-01-preview "HTTP/1.1 200 OK"


100%|██████████| 1/1 [00:00<00:00,  4.02it/s]


{
│   'context': {
│   │   'question_to_rewrite': [
│   │   │   Document(
│   │   │   │   metadata={
│   │   │   │   │   'source': 'docs/30-development/development-c2fec62.md',
│   │   │   │   │   'file_path': 'docs/30-development/development-c2fec62.md',
│   │   │   │   │   'file_name': 'development-c2fec62.md',
│   │   │   │   │   'file_type': '.md',
│   │   │   │   │   'start_index': 2230
│   │   │   │   },
│   │   │   │   page_content='</td>\n</tr>\n<tr>\n<td valign="top">\n\n**Additional Information**\n\n</td>\n<td valign="top">\n\n[Comparison: SAP BTP, Kyma Runtime and SAP BTP, Cloud Foundry Runtime](https://help.sap.com/docs/btp/comparison-btp-runtimes/runtime-comparison?version=Cloud)\n\n</td>\n<td valign="top">\n\n[Comparison: SAP BTP, Kyma Runtime and SAP BTP, Cloud Foundry Runtime](https://help.sap.com/docs/btp/comparison-btp-runtimes/runtime-comparison?version=Cloud)\n\n</td>\n<td valign="top">\n\n[Development in the ABAP Environment](https://help.sap.com/docs/btp/sap-business-technology-platform/development-in-abap-environment?version=Cloud)\n\n</td>\n</tr>\n<tr>\n<td valign="top">\n\n**Shared Benefits**\n\n</td>\n<td valign="top" colspan="3">\n\n-   No infrastructure vendor lock-in\n\n-   Build scalable multitenancy business applications \\(SaaS\\)\n\n-   Out-of-the-box consumption of SAP and hyperscaler services\n\n-   Built on industry standards and open technology\n\n\n\n\n</td>\n</tr>\n<tr>\n<td valign="top">\n\n**Good For**'
│   │   │   ),
│   │   │   Document(
│   │   │   │   metadata={
│   │   │   │   │   'source': 'docs/30-development/development-c2fec62.md',
│   │   │   │   │   'file_path': 'docs/30-development/development-c2fec62.md',
│   │   │   │   │   'file_name': 'development-c2fec62.md',
│   │   │   │   │   'file_type': '.md',
│   │   │   │   │   'start_index': 1267
│   │   │   │   },
│   │   │   │   page_content='</td>\n<td valign="top">\n\n-   Take full advantage of the advanced features and rich ecosystem of Kubernetes\n-   Free choice of programming languages and models \\(containerized deployments\\)\n-   Combines microservices and serverless functions\n-   Built-in, managed service mesh based on Istio, and other cloud-native open-source modules to reduce the development effort\n-   Built-in, managed event mesh\n-   Managed infrastructure: day-2 operations, security patches, and updates\n-   Full administrator access\n-   Refined horizontal and vertical automatic scalability\n-   Dedicated application runtime\n-   Zero downtime infrastructure setup by default\n-   Support for CAP – an opinionated business app development framework\n-   Support for on-premise connectivity\n\n\n\n</td>\n<td valign="top">\n\n-   ABAP programming language\n-   Fast prototyping with ABAP RESTful Programming Model \\(RAP\\)\n-   Integrated development lifecycle\n-   Reuse existing on-prem ABAP assets'
│   │   │   ),
│   │   │   Document(
│   │   │   │   metadata={
│   │   │   │   │   'source': 'docs/70-getting-support/comparison-between-the-operating-models-of-kyma-and-cloud-foundry-runtimes-3978f94.md',
│   │   │   │   │   'file_path': 'docs/70-getting-support/comparison-between-the-operating-models-of-kyma-and-cloud-foundry-runtimes-3978f94.md',
│   │   │   │   │   'file_name': 'comparison-between-the-operating-models-of-kyma-and-cloud-foundry-runtimes-3978f94.md',
│   │   │   │   │   'file_type': '.md',
│   │   │   │   │   'start_index': 0
│   │   │   │   },
│   │   │   │   page_content='<!-- loio3978f94fccd24bf3b013a1a6ed25f55d -->\n\n# Comparison between the Operating Models of Kyma and Cloud Foundry Runtimes\n\nThis operating model clearly defines the separation of tasks between SAP and the customer. It\'s relevant for all phases of a project for both environments.\n\n\n\nThe responsibilities for operating the Cloud Foundry and Kyma runtimes are described in the following service catalog.\n\n**Service Catalog**\n\n\n<table>\n<tr>\n<th valign="top">\n\nProcess\n\n</th>\n<th valign="top">\n\nTask\n\n</th>\n<th v

In [25]:
rewrite_retrieve_read_chain = (
        rewriter
        | {
            "context": {"question_to_rewrite": RunnablePassthrough() | retrieve_chunks},
            "question": RunnablePassthrough() | rewritten_query,
        }
        | prompt
        #| llm
        #| StrOutputParser()
    )

intermediate = rewrite_retrieve_read_chain.invoke(simple_query)

 

console.print(Pretty(intermediate, indent_guides=True))




2026-03-23 05:45:08,615 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/da74d08d2f277477/chat/completions?api-version=2025-03-01-preview "HTTP/1.1 200 OK"
Query after rewrite: 
 To enhance the query for database similarity retrieval and leverage a large language model to answer the question effectively, it's important to provide context and specify the criteria for comparison. Here's a refined query:

---

**Query:**

"I am evaluating cloud-native platforms for application deployment and management. Could you provide a detailed comparison between Cloud Foundry and Kyma, focusing on aspects such as scalability, ease of use, integration capabilities, community support, and cost-effectiveness? Additionally, are there other platforms that might be more suitable for specific use cases?"

---

**Explanation:**

1. **Contextualization**: The query specifies the purpose of the evaluation—application deployment and managemen

  0%|          | 0/1 [00:00<?, ?it/s]

2026-03-23 05:45:08,902 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/d12a770eac7c91c9/embeddings?api-version=2025-03-01-preview "HTTP/1.1 200 OK"


100%|██████████| 1/1 [00:00<00:00,  3.52it/s]


ChatPromptValue(
│   messages=[
│   │   HumanMessage(
│   │   │   content='\n    Answer the question in detail and as truthfully as possible based only on the provided context. If you\'re unsure of the question or answer, say "Sorry, I don\'t know".\n    <context>\n    {\'question_to_rewrite\': [Document(metadata={\'source\': \'docs/30-development/development-c2fec62.md\', \'file_path\': \'docs/30-development/development-c2fec62.md\', \'file_name\': \'development-c2fec62.md\', \'file_type\': \'.md\', \'start_index\': 2230}, page_content=\'</td>\\n</tr>\\n<tr>\\n<td valign="top">\\n\\n**Additional Information**\\n\\n</td>\\n<td valign="top">\\n\\n[Comparison: SAP BTP, Kyma Runtime and SAP BTP, Cloud Foundry Runtime](https://help.sap.com/docs/btp/comparison-btp-runtimes/runtime-comparison?version=Cloud)\\n\\n</td>\\n<td valign="top">\\n\\n[Comparison: SAP BTP, Kyma Runtime and SAP BTP, Cloud Foundry Runtime](https://help.sap.com/docs/btp/comparison-btp-runtimes/runtime-comparison?version=Cloud)\\n\\n</td>\\n<td valign="top">\\n\\n[Development in the ABAP Environment](https://help.sap.com/docs/btp/sap-business-technology-platform/development-in-abap-environment?version=Cloud)\\n\\n</td>\\n</tr>\\n<tr>\\n<td valign="top">\\n\\n**Shared Benefits**\\n\\n</td>\\n<td valign="top" colspan="3">\\n\\n-   No infrastructure vendor lock-in\\n\\n-   Build scalable multitenancy business applications \\\\(SaaS\\\\)\\n\\n-   Out-of-the-box consumption of SAP and hyperscaler services\\n\\n-   Built on industry standards and open technology\\n\\n\\n\\n\\n</td>\\n</tr>\\n<tr>\\n<td valign="top">\\n\\n**Good For**\'), Document(metadata={\'source\': \'docs/30-development/development-c2fec62.md\', \'file_path\': \'docs/30-development/development-c2fec62.md\', \'file_name\': \'development-c2fec62.md\', \'file_type\': \'.md\', \'start_index\': 1267}, page_content=\'</td>\\n<td valign="top">\\n\\n-   Take full advantage of the advanced features and rich ecosystem of Kubernetes\\n-   Free choice of programming languages and models \\\\(containerized deployments\\\\)\\n-   Combines microservices and serverless functions\\n-   Built-in, managed service mesh based on Istio, and other cloud-native open-source modules to reduce the development effort\\n-   Built-in, managed event mesh\\n-   Managed infrastructure: day-2 operations, security patches, and updates\\n-   Full administrator access\\n-   Refined horizontal and vertical automatic scalability\\n-   Dedicated application runtime\\n-   Zero downtime infrastructure setup by default\\n-   Support for CAP – an opinionated business app development framework\\n-   Support for on-premise connectivity\\n\\n\\n\\n</td>\\n<td valign="top">\\n\\n-   ABAP programming language\\n-   Fast prototyping with ABAP RESTful Programming Model \\\\(RAP\\\\)\\n-   Integrated development lifecycle\\n-   Reuse existing on-prem ABAP assets\'), Document(metadata={\'source\': \'docs/70-getting-support/operating-model-in-the-kyma-environment-862b96b.md\', \'file_path\': \'docs/70-getting-support/operating-model-in-the-kyma-environment-862b96b.md\', \'file_name\': \'operating-model-in-the-kyma-environment-862b96b.md\', \'file_type\': \'.md\', \'start_index\': 14346}, page_content=\'[Comparison between the Operating Models of Kyma and Cloud Foundry Runtimes](comparison-between-the-operating-models-of-kyma-and-cloud-foundry-runtimes-3978f94.md "This operating model clearly defines the separation of tasks between SAP and the customer. It\\\'s relevant for all phases of a project for both environments.")\\n\\n[SLAs for Cloud Services](https://www.sap.com/about/trust-center/agreements/cloud/cloud-services.html?search=Service%20Level%20Agreement&sort=latest_desc)\'), Document(metadata={\'source\': \'docs/70-getting-support/operating-model-in-the-cloud-foundry-environment-de55b6e.md\', \'file_path\': \'docs/70-getting-support/operating-model-in-the-cloud-foundry-environment-de55b6e.md\', \'file_name\': \'operating-model-in-the-cloud-foun

#### Full function

In [19]:
def invoke_query_with_rewrite(llm, simple_query, prompt, retriever):
    rewrite_template = """Provide a better query for
    the database similarity retrieval and large language model to answer the given question.
    Question: {question_to_rewrite}
    Answer:"""
    rewrite_prompt = ChatPromptTemplate.from_template(rewrite_template)

    def rewritten_query(rewritten_query: str):
        print("Query after rewrite: ", rewritten_query)
        return rewritten_query

    rewriter = rewrite_prompt | llm | StrOutputParser()

    rewrite_retrieve_read_chain = (
        rewriter
        | {
            "context": {"question_to_rewrite": RunnablePassthrough() | retrieve_chunks},
            "question": RunnablePassthrough() | rewritten_query,
        }
        | prompt
        | llm
        | StrOutputParser()
    )

    rewritten_result = rewrite_retrieve_read_chain.invoke(simple_query)
    print("QA result after query rewrite: ", rewritten_result)

In [27]:
invoke_query_with_rewrite(
    llm=llm, 
    simple_query=simple_query, 
    prompt=prompt,
    retriever=retrieve_chunks
)

2026-03-23 05:47:43,104 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/da74d08d2f277477/chat/completions?api-version=2025-03-01-preview "HTTP/1.1 200 OK"
Query after rewrite:  To improve the query for database similarity retrieval and leverage a large language model to answer the question effectively, it's important to provide more context and specify the criteria or aspects you're interested in comparing. Here's a refined query:

---

**Query:**

"I'm evaluating cloud-native platforms and need a comparison between Cloud Foundry and Kyma. Could you provide insights on their key features, use cases, advantages, and potential drawbacks? Additionally, are there other platforms I should consider for building and managing cloud-native applications?"

---

This query provides a clear context and specifies the aspects you are interested in, which will help in retrieving more relevant information and generating a comprehen

  0%|          | 0/1 [00:00<?, ?it/s]

2026-03-23 05:47:43,350 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/d12a770eac7c91c9/embeddings?api-version=2025-03-01-preview "HTTP/1.1 200 OK"


100%|██████████| 1/1 [00:00<00:00,  4.11it/s]


2026-03-23 05:47:48,519 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/da74d08d2f277477/chat/completions?api-version=2025-03-01-preview "HTTP/1.1 200 OK"
QA result after query rewrite:  To compare Cloud Foundry and Kyma, let's delve into their key features, use cases, advantages, and potential drawbacks:

### Cloud Foundry

**Key Features:**
- **Polyglot Environment:** Supports multiple programming languages and frameworks, allowing developers to use the best tools for their projects.
- **Buildpacks:** Offers a variety of buildpacks, including community and self-developed options, to streamline application deployment.
- **SAP Integration:** Integrates with SAP HANA extended application services, advanced model.
- **Lifecycle Management:** Provides tools for starting, stopping, scaling, and configuring applications.
- **Cloud-Native Capabilities:** Emphasizes containerization and multitenancy for resilient and effic

## Advanced RAG: Mixed data

In [4]:
from gen_ai_hub.proxy.langchain.openai import ChatOpenAI
from gen_ai_hub.proxy.langchain.openai import OpenAIEmbeddings
from gen_ai_hub.proxy.core.proxy_clients import get_proxy_client
from langchain_core.rate_limiters import InMemoryRateLimiter

# Define a function to define the  chat llm and embedding model to be used
def create_llm_and_embeddings(llm_model,embedding_model):
    
    # Get the proxy client for the AI Core service
    proxy_client = get_proxy_client("gen-ai-hub")
     
    rate_limiter = InMemoryRateLimiter(
        requests_per_second=0.5,  # We can only make a request once every 5 seconds
        check_every_n_seconds=0.1,  # Wake up every 100 ms to check whether allowed to make a request,
        max_bucket_size=10,  # Controls the maximum burst size.
    )

    llm = ChatOpenAI(
        proxy_model_name=llm_model,
        proxy_client=proxy_client,
        temperature=0.1,
        rate_limiter=rate_limiter,
    )

    embeddings = OpenAIEmbeddings(
        proxy_model_name=embedding_model,
        proxy_client=proxy_client,
        show_progress_bar=True,
    )
    return llm, embeddings

# Initial the model
LLM_MODEL_NAME = 'gpt-4o'
EMBEDDINGS_MODEL_NAME ='text-embedding-3-large'
llm, embeddings = create_llm_and_embeddings(
    llm_model=LLM_MODEL_NAME,
    embedding_model=EMBEDDINGS_MODEL_NAME,
)


# Define a function to check whether a table exists or not
def check_if_exists(table_name, schema_name="USR_7K4HHI6O79L4LB691X7CN6MUQ"):
    connection_to_hana = init_env.get_connection_to_hana_db()
    cursor = connection_to_hana.cursor()

    # Check if the table exists
    check_table_query ="""
    SELECT COUNT(*)
    FROM TABLES
    WHERE SCHEMA_NAME = ? AND TABLE_NAME = ?
    """

    cursor.execute(check_table_query, (schema_name, table_name))
    return cursor.fetchone()[0] > 0

# Define a function to remove existing table
def teardown_hana_table(table_name):
    
    exists = check_if_exists(table_name=table_name)
    if exists is False:
        log.info(f"Table {table_name} does not exsit. Nothing to clean up.")
        return
    
    try:
        connection_to_hana = init_env.get_connection_to_hana_db()
        cursor = connection_to_hana.cursor()
        log.info(f"Dropping table {table_name}")
        cursor.execute(f"DROP TABLE {table_name}")
        cursor.close()
        log.info(f"Table {table_name} dropped successfully.")
    except Exception as e:
        log.error(type(e))
        log.error(f"Error dropping table: {str(e)}")

2026-03-27 06:30:47,330 - numexpr.utils - INFO - NumExpr defaulting to 8 threads.


### Prepare Data

In [4]:
cities = [
    {"city_name": "Toronto", "population": 2930000, "country": "Canada"},
    {"city_name": "Tokyo", "population": 13960000, "country": "Japan"},
    {"city_name": "Berlin", "population": 3645000, "country": "Germany"},
]


STRUCTURED_DATA_TABLE_NAME = "CITY_STATS"
VECTOR_EMBEDDINGS_TABLE_NAME = "CITY_WIKI_PAGES_EMBEDDINGS"

#### Prepare Data : Structured data

In [92]:
def ingest_structured_data(cities):
    try:
        log.info("Start ingesting structured data.")
        teardown_hana_table(STRUCTURED_DATA_TABLE_NAME)
        connection_to_hana = init_env.get_connection_to_hana_db()
        cur = connection_to_hana.cursor()

        cur.execute(
            f"CREATE TABLE {STRUCTURED_DATA_TABLE_NAME} (CITY_NAME NCHAR(16) PRIMARY KEY, POPULATION INTEGER, COUNTRY NCHAR(16))"
        )

        sql = f"INSERT INTO {STRUCTURED_DATA_TABLE_NAME} (CITY_NAME, POPULATION, COUNTRY) VALUES (:city_name, :population, :country)"
        for city in cities:
            cur.execute(
                sql,
                {
                    "city_name": city["city_name"],
                    "population": city["population"],
                    "country": city["country"],
                },
            )

        log.info("Table with structured data created:")
        cur.execute(f"SELECT * FROM {STRUCTURED_DATA_TABLE_NAME}")
        log.info(cur.fetchall())
        cur.close()
        log.info("Structured data ingested successfully.")
    
    except Exception as e:
        log.error(f"An error occurred while ingesting structured data: {str(e)}")
        teardown_hana_table(STRUCTURED_DATA_TABLE_NAME)
        sys.exit()

In [93]:
# Execute
ingest_structured_data(cities)

2026-03-25 08:32:36,109 - __main__ - INFO - Start ingesting structured data.
2026-03-25 08:32:36,474 - __main__ - INFO - Dropping table CITY_STATS
2026-03-25 08:32:36,483 - __main__ - INFO - Table CITY_STATS dropped successfully.
2026-03-25 08:32:36,694 - __main__ - INFO - Table with structured data created:
2026-03-25 08:32:36,705 - __main__ - INFO - [('Toronto', 2930000, 'Canada'), ('Tokyo', 13960000, 'Japan'), ('Berlin', 3645000, 'Germany')]
2026-03-25 08:32:36,706 - __main__ - INFO - Structured data ingested successfully.


#### Prepare Data : Unstructured data

##### Step by step

In [95]:
# Load from Wikipedia
from langchain_community.document_loaders import WikipediaLoader # https://docs.langchain.com/oss/python/integrations/document_loaders/wikipedia

wiki_docs = [
    WikipediaLoader(
        query=row["city_name"], 
        doc_content_chars_max=1000000, #(optional): default=4000. The maximum number of characters for the document content.
        load_max_docs=1
    ).load()[0]
    for row in cities
]
log.info(f"Found {len(wiki_docs)} documents from Wikipedia.")


2026-03-25 08:33:46,893 - __main__ - INFO - Found 3 documents from Wikipedia.


In [ ]:
# Test
console.print(Pretty(wiki_docs, indent_guides=True)) 

In [ ]:
# Test
print(wiki_docs[1].page_content)

In [98]:
from langchain_text_splitters.character import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# Define a function to chunk fetched document
def split_docs_into_chunks(
    documents: list[Document], chunk_size: int = 1000, chunk_overlap: int = 100
):
    """
    Splits a list of documents into chunks of specified size with overlap.

    Args:
        documents (list[Document]): The list of documents to be split into chunks.
        chunk_size (int, optional): The size of each chunk. Defaults to 1000.
        chunk_overlap (int, optional): The overlap between consecutive chunks. Defaults to 100.

    Returns:
        list[list[Document]]: A list of chunks, where each chunk is a list of documents.

    """
    try:
        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            length_function=len,
            add_start_index=True,
        )
        chunks = text_splitter.split_documents(documents)
        log.info(f"Split {len(documents)} documents into {len(chunks)} chunks.")

        return chunks
    except Exception as e:
        log.error(f"An error occurred while splitting documents into chunks: {str(e)}")
        raise

In [99]:
#Test
chunks = split_docs_into_chunks(documents=wiki_docs)

2026-03-25 08:34:31,494 - __main__ - INFO - Split 3 documents into 411 chunks.


##### Full function

In [104]:
from langchain_text_splitters.character import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# Define a function to chunk fetched document
def split_docs_into_chunks(
    documents: list[Document], chunk_size: int = 1000, chunk_overlap: int = 100
):
    """
    Splits a list of documents into chunks of specified size with overlap.

    Args:
        documents (list[Document]): The list of documents to be split into chunks.
        chunk_size (int, optional): The size of each chunk. Defaults to 1000.
        chunk_overlap (int, optional): The overlap between consecutive chunks. Defaults to 100.

    Returns:
        list[list[Document]]: A list of chunks, where each chunk is a list of documents.

    """
    try:
        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            length_function=len,
            add_start_index=True,
        )
        chunks = text_splitter.split_documents(documents)
        log.info(f"Split {len(documents)} documents into {len(chunks)} chunks.")

        return chunks
    except Exception as e:
        log.error(f"An error occurred while splitting documents into chunks: {str(e)}")
        raise


from langchain_hana import HanaDB 
from langchain_community.document_loaders import WikipediaLoader
# Define the full function to fetch document from Wikipedia and then ingest to HanaDB 
def ingest_unstructured_data(cities):
    try:
        
        # Load from Wikipedia
        log.info("Start ingesting unstructured data.")
        log.info("Start fetching documents from Wikipedia")
        wiki_docs = [
            WikipediaLoader(
                query=row["city_name"], 
                doc_content_chars_max=1000000, #(optional): default=4000. The maximum number of characters for the document content.
                load_max_docs=1
            ).load()[0]
            for row in cities
        ]
        log.info(f"Found {len(wiki_docs)} documents from Wikipedia.")
         

        # Split the documents into chunks
        chunks = split_docs_into_chunks(documents=wiki_docs)

        # Create the HanaDB object
        connection_to_hana = init_env.get_connection_to_hana_db()
        db = HanaDB(
            embedding=embeddings,
            connection=connection_to_hana,
            table_name=VECTOR_EMBEDDINGS_TABLE_NAME,
        )

        # Delete already existing documents from the table
        log.info("Cleaning up table with vector embeddings.")
        db.delete(filter={})
        log.info("Table cleaned up successfully.")
         

        # add the loaded document chunks to the HANA DB
        log.info("Adding the loaded document chunks to the HANA DB")
        db.add_documents(chunks)
        log.info("Unstructured data ingested successfully.")
    
    except Exception as e:
        log.error(f"Ingesting unstructured data failed: {str(e)}")
        sys.exit()

In [ ]:
# Execute
ingest_unstructured_data(cities=cities)

##### Test

In [ ]:
from langchain_hana import HanaDB 
connection_to_hana = init_env.get_connection_to_hana_db()
db = HanaDB(
    embedding=embeddings,
    connection=connection_to_hana,
    table_name=VECTOR_EMBEDDINGS_TABLE_NAME,
)

retriever = db.as_retriever(
    search_kwargs={"k": 1}
)

from langchain.chains import RetrievalQA
qa = RetrievalQA.from_llm(
    llm=llm, 
    retriever=retriever, 
)


In [30]:
 
query = "What is number of the Fortune Global 500 companies were headquartered in Tokyo in 2025? Which area is their prefered place in the city?"


answer = llm.invoke(query)
log.info("Answer without RAG:")
print(answer.content)

answer = qa.invoke(query)
log.info("Answer with ingested Wikipedia data:")
print(answer["result"])

2026-03-26 05:31:18,070 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/da74d08d2f277477/chat/completions?api-version=2025-03-01-preview "HTTP/1.1 200 OK"
2026-03-26 05:31:18,072 - __main__ - INFO - Answer without RAG:
As of my last update in October 2023, I don't have specific data for the year 2025. However, Tokyo has consistently been a major hub for Fortune Global 500 companies. In recent years, many of these companies have been headquartered in areas like Marunouchi, Shinjuku, and Minato, which are known for their business districts and proximity to government offices, financial institutions, and other corporate entities.

For the most accurate and up-to-date information, you would need to consult the latest Fortune Global 500 list or business reports specific to Tokyo in 2025.


  0%|          | 0/1 [00:00<?, ?it/s]

2026-03-26 05:31:18,283 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/d12a770eac7c91c9/embeddings?api-version=2025-03-01-preview "HTTP/1.1 200 OK"


100%|██████████| 1/1 [00:00<00:00,  4.73it/s]


2026-03-26 05:31:19,017 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/da74d08d2f277477/chat/completions?api-version=2025-03-01-preview "HTTP/1.1 200 OK"
2026-03-26 05:31:19,018 - __main__ - INFO - Answer with ingested Wikipedia data:
In 2025, 26 of the Fortune Global 500 companies were headquartered in Tokyo. Many of them, around 20, are based in the Marunouchi area.


### Define Agent 

#### Create Retriever 

In [ ]:
from langchain_community.vectorstores.hanavector import HanaDB
def create_retriever(embeddings):
    assert (
        STRUCTURED_DATA_TABLE_NAME
    ), "STRUCTURED_DATA_TABLE_NAME is not defined. Please define it before proceeding."

    try:
        connection_to_hana = init_env.get_connection_to_hana_db()
        vector_db = HanaDB(
            embedding=embeddings,
            connection=connection_to_hana,
            table_name=VECTOR_EMBEDDINGS_TABLE_NAME,
        )
        return vector_db.as_retriever(search_kwargs={"k": 10})
    
    except Exception as e:
        log.error(f"An error occurred while creating the retriever: {str(e)}")
        sys.exit()


In [ ]:
# Create retriver
STRUCTURED_DATA_TABLE_NAME = "CITY_STATS"
VECTOR_EMBEDDINGS_TABLE_NAME = "CITY_WIKI_PAGES_EMBEDDINGS"
retriever = create_retriever(embeddings=embeddings) 


/tmp/ipykernel_48635/1627048354.py:9: LangChainDeprecationWarning: This class is deprecated and will be removed in a future version. Please use HanaDB from the langchain_hana package instead. See https://github.com/SAP/langchain-integration-for-sap-hana-cloud for details.
  vector_db = HanaDB(


#### Define Tool

In [7]:
from langchain.tools import BaseTool
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain.callbacks.manager import (
    AsyncCallbackManagerForToolRun,
    CallbackManagerForToolRun,
)
from langchain_core.language_models import LanguageModelLike
from langchain_core.vectorstores import VectorStoreRetriever
from typing import Optional


class RAGTool(BaseTool):
    name : str ="rag_tool"
    description: str = """
        Useful information about Cities from Wikipedia.
        Input: A question about a city.
        Output: The answer to the question."""
    llm: LanguageModelLike
    retriever: VectorStoreRetriever

    def _run(
        self, query: str, run_manager: Optional[CallbackManagerForToolRun] = None
    ) -> str:
        """Use the tool."""
        system_prompt = (
            "Use the given context to answer the question. "
            "If you don't know the answer, say you don't know. "
            "Use three sentence maximum and keep the answer concise. "
            "Context: {context}"
        )
        prompt = ChatPromptTemplate.from_messages(
            [
                ("system", system_prompt),
                ("human", "{input}"),
            ]
        )

        question_answer_chain = create_stuff_documents_chain(self.llm, prompt)
        chain = create_retrieval_chain(self.retriever, question_answer_chain)

        result = chain.invoke({"input": query})

        return result["answer"]

    async def _arun(
        self, query: str, run_manager: Optional[AsyncCallbackManagerForToolRun] = None
    ) -> str:
        """Use the tool asynchronously."""
        raise NotImplementedError("custom_search does not support async")

#### Define Agent Prompt

In [52]:
SQL_AGENT_PREFIX = """
 

You are an agent designed to interact with a SQL database. 
You also have access to an external knowledge tool called `rag_tool`. 

## CRITICAL RULES FOR HYBRID SQL + RAG REASONING

### 1. SQL ALWAYS COMES FIRST WHEN THE QUESTION REQUIRES DATABASE-DERIVED ENTITIES.
If answering the question requires identifying *which entity* (city, person, product, etc.) from the database satisfies a condition (e.g., “the city with the highest population”),  
**you MUST first run a SQL query to compute that entity**, even if the final question is not directly SQL-related.

You are STRICTLY FORBIDDEN from guessing or using pre‑trained world knowledge to determine such entities.  
You MUST derive them from SQL results.

### 2. AFTER obtaining the SQL result, if the user asks for information NOT found inside the database (e.g., culture, history, arts, description),  
then you MUST call `rag_tool`, and the input MUST be based on the SQL result.

### 3. NEVER skip the SQL step if SQL is needed to compute intermediate values.

### 4. If SQL is not needed at all AND the question cannot be answered using the database,
then, and only then, you may directly use `rag_tool`.

---

## GENERAL SQL USAGE RULES
- Create syntactically correct {dialect} queries.
- Limit results to at most {top_k}, unless specified otherwise.
- Only select relevant columns.
- Double-check all queries before execution.
- On SQL execution error: rewrite and retry.
- Do NOT execute any DML (INSERT/UPDATE/DELETE/DROP).
- Do NOT use prior knowledge; rely ONLY on:
  * SQL results, or
  * rag_tool output.

---

## TOOL USAGE FORMAT (MANDATORY)
When using tools, follow the exact ReAct format:

Thought: <your reasoning>  
Action: <tool_name>  
Action Input: <input without markdown backticks>

For the final answer:

Final Answer: <your answer>  
Explanation: <how SQL and/or RAG were used>

---

## FALLBACK RULE
If SQL is not applicable AND no intermediate entity must be computed AND the question requires external knowledge,  
use `rag_tool`.

Do NOT output "I don't know" unless ALL tools fail and absolutely no information is available.

---

## EXAMPLES OF SQL → RAG CHAINING

Example:
User: "Tell me about the arts and culture of the city with the highest population."

Correct reasoning:
1. SQL query to find which city in the database has the highest population.
2. Use rag_tool to describe the arts and culture of THAT city (not a guessed one).

Incorrect:
- Guessing the city based on general world knowledge.
- Calling rag_tool without first running SQL.

---

   
### Examples of Final Answer:  
   
<example_1> 
Final Answer: There were 27,437 people who visited Paris in 2020.  
Explanation: I queried the `tourism` table for the `visitors` column where the city is 'Paris' and the date starts with '2020'. The query returned a list of tuples with the number of visitors for each month in 2020. To answer the question, I took the sum of all the visitors in the list, which is 27,437. I used the following query:  
```sql  
SELECT [visitors] FROM tourism WHERE city = 'Paris' AND date LIKE '2020%'  
```  
</example_1> 

<example_2>  
Final Answer: The average hotel price in Tokyo in 2021 was $322.5.  
Explanation: I queried the `hotel_prices` table for the average `price` where the city is 'Tokyo' and the year is '2021'. The SQL query used is:  
```sql  
SELECT AVG(price) AS average_price FROM hotel_prices WHERE city = 'Tokyo' AND year = '2021'  
```  
This query calculates the average price of all hotel stays in Tokyo for the year 2021, which is $322.5.  
</example_2>

<example_3>
Final Answer: There were 150 unique tourists who visited New York in 2024.  
Explanation: To find the number of unique tourists who visited New York in 2024, I used the following SQL query:  
```sql  
SELECT COUNT(DISTINCT tourist_id) FROM visits WHERE city = 'New York' AND visit_date BETWEEN '2024-01-01' AND '2024-12-31'  
```  
This query counts the distinct `tourist_id` entries within the `visits` table for the year 2024, resulting in 150 unique tourists.  
</example_3>

<example_4> 
Final Answer: The most popular landmark in Rome is the Colosseum.  
Explanation: I queried the `landmarks` table to find the name of the most popular landmark using the following SQL query:  
```sql  
SELECT TOP 1 name FROM landmarks WHERE city = 'Rome' ORDER BY popularity DESC  
```  
This query selects the landmark name from the `landmarks` table and orders the results by the `popularity` column in descending order. The `TOP 1` clause ensures that only the most popular landmark is returned, which is the 'Colosseum'.
</example_4>
    
"""

#### Define Agent

In [53]:
from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits.sql.base import create_sql_agent
from langchain_community.vectorstores.hanavector import HanaDB
from langchain_community.agent_toolkits.sql.toolkit import SQLDatabaseToolkit
from langchain.agents.types import AgentType

def create_agent(llm, retriever):
    try:
        rag_tool = RAGTool(llm=llm, retriever=retriever)
        hana_db_uri = init_env.get_connection_to_hana_string()
        db = SQLDatabase.from_uri(hana_db_uri)
        agent_executor = create_sql_agent(
            llm=llm,
            toolkit=SQLDatabaseToolkit(db=db, llm=llm),
            verbose=True,
            agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
            extra_tools=[rag_tool],
            prefix=SQL_AGENT_PREFIX,
            agent_executor_kwargs={"handle_parsing_errors": True},
        )
        return agent_executor
    except Exception as e:
        log.error(f"An error occurred while creating the agent: {str(e)}")
        sys.exit()

#### Run RAGRAG 

In [54]:
# Create agent
agent_executor = create_agent(llm, retriever)

In [69]:
query = "Can you give me the country corresponding to each city?"
anwser=agent_executor.invoke({"input": query})
print(anwser["output"])



> Entering new SQL Agent Executor chain...


2026-03-27 08:51:50,810 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/da74d08d2f277477/chat/completions?api-version=2025-03-01-preview "HTTP/1.1 200 OK"
Action: sql_db_list_tables  
Action Input: ""  city_stats, city_wiki_pages_embeddings, git_docs, pdf_docs, table2503, test_embedding_table2026-03-27 08:51:51,543 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/da74d08d2f277477/chat/completions?api-version=2025-03-01-preview "HTTP/1.1 200 OK"
To find the country corresponding to each city, I should check the schema of the `city_stats` table, as it likely contains information about cities. I will query the schema of this table to identify the relevant columns.

Action: sql_db_schema  
Action Input: city_stats  
CREATE TABLE city_stats (
	city_name NCHAR(16) NOT NULL, 
	population INTEGER, 
	country NCHAR(16), 
	CONSTRAINT "_SYS_TREE_CS_#887802_#0

In [70]:
query = "What is number of the Fortune Global 500 companies were headquartered in Tokyo in 2025? Which area is their prefered place in the city?"

anwser=agent_executor.invoke({"input": query})
print(anwser["output"])



> Entering new SQL Agent Executor chain...
2026-03-27 08:52:18,022 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/da74d08d2f277477/chat/completions?api-version=2025-03-01-preview "HTTP/1.1 200 OK"


Action: sql_db_list_tables  
Action Input: ""city_stats, city_wiki_pages_embeddings, git_docs, pdf_docs, table2503, test_embedding_table2026-03-27 08:52:19,273 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/da74d08d2f277477/chat/completions?api-version=2025-03-01-preview "HTTP/1.1 200 OK"
The table names don't clearly indicate which one might contain information about Fortune Global 500 companies. I should check the schema of all tables to identify the relevant one for this query.

Action: sql_db_schema  
Action Input: city_stats, city_wiki_pages_embeddings, git_docs, pdf_docs, table2503, test_embedding_table  
CREATE TABLE city_stats (
	city_name NCHAR(16) NOT NULL, 
	population INTEGER, 
	country NCHAR(16), 
	CONSTRAINT "_SYS_TREE_CS_#887802_#0_#P0" PRIMARY KEY (city_name)
)

/*
3 rows from city_stats table:
city_name	population	country
Toronto	2930000	Canada
Tokyo	13960000	Japan
Berlin	3645000	Germany
*/


CREAT

  0%|          | 0/1 [00:00<?, ?it/s]

2026-03-27 08:52:21,769 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/d12a770eac7c91c9/embeddings?api-version=2025-03-01-preview "HTTP/1.1 200 OK"


100%|██████████| 1/1 [00:00<00:00,  2.76it/s]


2026-03-27 08:52:23,032 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/da74d08d2f277477/chat/completions?api-version=2025-03-01-preview "HTTP/1.1 200 OK"
In 2025, 26 Fortune Global 500 companies were headquartered in Tokyo. Notably, around 20 of these companies, including MUFG, Mitsubishi Corp., and Hitachi, are based in the Marunouchi area.2026-03-27 08:52:23,698 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/da74d08d2f277477/chat/completions?api-version=2025-03-01-preview "HTTP/1.1 200 OK"
I now know the final answer.

Final Answer: In 2025, there were 26 Fortune Global 500 companies headquartered in Tokyo. The preferred area for these companies in the city is the Marunouchi area, where around 20 of them, including MUFG, Mitsubishi Corp., and Hitachi, are based.  
Explanation: Since the database did not contain information about Fortune Globa

In [71]:
query = "What is  the history of Beijing?"

anwser=agent_executor.invoke({"input": query})
print(anwser["output"])



> Entering new SQL Agent Executor chain...


2026-03-27 08:54:39,280 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/da74d08d2f277477/chat/completions?api-version=2025-03-01-preview "HTTP/1.1 200 OK"
Thought: Since the question is about the history of Beijing, which is not typically stored in a database, I should first check if there are any relevant tables that might contain historical information. However, based on the rules, if the question is not related to database-derived entities, I should directly use the `rag_tool` for external knowledge.

Action: rag_tool
Action Input: What is the history of Beijing?


  0%|          | 0/1 [00:00<?, ?it/s]

2026-03-27 08:54:40,296 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/d12a770eac7c91c9/embeddings?api-version=2025-03-01-preview "HTTP/1.1 200 OK"


100%|██████████| 1/1 [00:00<00:00,  4.18it/s]


2026-03-27 08:54:40,961 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/da74d08d2f277477/chat/completions?api-version=2025-03-01-preview "HTTP/1.1 200 OK"
I don't know.2026-03-27 08:54:41,869 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/da74d08d2f277477/chat/completions?api-version=2025-03-01-preview "HTTP/1.1 200 OK"
Since the `rag_tool` did not provide information about the history of Beijing, I will attempt to use the SQL database to see if there are any tables that might contain historical information about cities. However, typically such information is not stored in a database, so this might not yield results.

Action: sql_db_list_tables
Action Input: 
city_stats, city_wiki_pages_embeddings, git_docs, pdf_docs, table2503, test_embedding_table2026-03-27 08:54:43,422 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.m

In [72]:


query = "Tell me about the arts and culture of the city with the highest population."


anwser=agent_executor.invoke({"input": query})
print(anwser["output"])



> Entering new SQL Agent Executor chain...


2026-03-27 08:55:08,029 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/da74d08d2f277477/chat/completions?api-version=2025-03-01-preview "HTTP/1.1 200 OK"
Action: sql_db_list_tables  
Action Input: ""city_stats, city_wiki_pages_embeddings, git_docs, pdf_docs, table2503, test_embedding_table2026-03-27 08:55:08,835 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/da74d08d2f277477/chat/completions?api-version=2025-03-01-preview "HTTP/1.1 200 OK"
To find the city with the highest population, I should first check the schema of the `city_stats` table to see if it contains population data.  
Action: sql_db_schema  
Action Input: city_stats  
CREATE TABLE city_stats (
	city_name NCHAR(16) NOT NULL, 
	population INTEGER, 
	country NCHAR(16), 
	CONSTRAINT "_SYS_TREE_CS_#887802_#0_#P0" PRIMARY KEY (city_name)
)

/*
3 rows from city_stats table:
city_name	pop

  0%|          | 0/1 [00:00<?, ?it/s]

2026-03-27 08:55:13,419 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/d12a770eac7c91c9/embeddings?api-version=2025-03-01-preview "HTTP/1.1 200 OK"


100%|██████████| 1/1 [00:00<00:00,  3.86it/s]


2026-03-27 08:55:15,228 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/da74d08d2f277477/chat/completions?api-version=2025-03-01-preview "HTTP/1.1 200 OK"
Tokyo is a vibrant cultural hub with numerous museums, art galleries, and libraries, such as the Tokyo National Museum, the National Museum of Modern Art, and the Mori Art Museum. The city is also known for its traditional and contemporary performing arts, with venues like the National Noh Theatre, Kabuki-za, and the New National Theatre Tokyo. Additionally, Tokyo's diverse cultural scene includes festivals, youth fashion in Harajuku, and otaku culture in Akihabara.2026-03-27 08:55:15,717 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/da74d08d2f277477/chat/completions?api-version=2025-03-01-preview "HTTP/1.1 200 OK"
I now know the final answer.  
Final Answer: Tokyo is a vibrant cultural hub w

In [73]:


query = "What is the total population of the whole country of Japan?"

anwser=agent_executor.invoke({"input": query})
print(anwser["output"])



> Entering new SQL Agent Executor chain...
2026-03-27 08:55:55,727 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/da74d08d2f277477/chat/completions?api-version=2025-03-01-preview "HTTP/1.1 200 OK"
Action: sql_db_list_tables  
Action Input: ""  city_stats, city_wiki_pages_embeddings, git_docs, pdf_docs, table2503, test_embedding_table2026-03-27 08:55:56,315 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/da74d08d2f277477/chat/completions?api-version=2025-03-01-preview "HTTP/1.1 200 OK"
I should check the schema of the `city_stats` table to see if it contains population data for cities in Japan. This will help me determine if I can calculate the total population of Japan by summing the populations of its cities.

Action: sql_db_schema  
Action Input: city_stats  
CREATE TABLE city_stats (
	city_name NCHAR(16) NOT NULL, 
	population INTEGER, 
	co

To find the total population of Japan, I need to sum the populations of all cities in Japan from the `city_stats` table.

Action: sql_db_query_checker  
Action Input: SELECT SUM(population) AS total_population FROM city_stats WHERE country = 'Japan'  2026-03-27 08:55:58,867 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/da74d08d2f277477/chat/completions?api-version=2025-03-01-preview "HTTP/1.1 200 OK"
```sql
SELECT SUM(population) AS total_population FROM city_stats WHERE country = 'Japan'
```2026-03-27 08:55:59,236 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/da74d08d2f277477/chat/completions?api-version=2025-03-01-preview "HTTP/1.1 200 OK"
Action: sql_db_query  
Action Input: SELECT SUM(population) AS total_population FROM city_stats WHERE country = 'Japan'  [(13960000,)]2026-03-27 08:56:00,270 - httpx - INFO - HTTP Request: POST https://ap